# Dynamic and Hybrid Conditioning for Compositional Image Retrieval

**Deep Learning Assignment 2026 — CelebA, frozen CLIP ViT-B/32**

Given a reference image $v_{ref}$ and attribute constraints $T^+ / T^-$ (e.g.
`+Smiling, -Blond_Hair`), retrieve test-split images that preserve the
reference's identity while satisfying every constraint.

**The idea in one sentence.** The assignment's own ground-truth rule (§3.1.1)
defines a correct answer entirely in terms of 40-bit *attribute codes* — not
embedding geometry — so we move the conditioning process out of CLIP's
embedding space and into a learned attribute space, where the fusion of
reference and constraints can be written down exactly and then scored directly.

| method | R@10 |
|---|---|
| zero-shot latent arithmetic (assignment baseline) | 0.106 |
| probe-direction composition, tuned $\gamma$ | 0.210 |
| CPAS-MLP fusion, cosine ranking (3 seeds) | 0.361 |
| **delivered: attribute-space score + CPAS-MLP** | **0.564** |

5.3× over the required zero-shot baseline. The notebook is the full codebase and the full
report. Every number is recomputed by the cells below from the cached
artifacts, which the cells rebuild when absent — with two exceptions that
need a GPU and are therefore run by scripts and loaded here: the combiner's
training history (§5) and the degraded-predictor sweep (§8). Both scripts
are in the repository and the cells print the command that produces them.

*Sections: 1 the criterion · 2 positioning · 3 the score · 4 architecture ·
5 training the fusion module · 6 experimental setup · 7 results · 8 the role of
the cosine term · 9 error decomposition and ceiling · 10 negative results ·
11 conclusions.*

## 0. Setup

Runs both on Colab (mount Drive, unzip CelebA to the local SSD) and locally
(repo root with `celeba/` and `celeba_evaluation.json` in place). All heavy
artifacts — cached features, fitted probes, the attribute head, the CPAS-MLP
checkpoint — are **loaded when present and computed when absent**, so a warm
re-run takes minutes while a cold run remains fully reproducible.

Approximate cold-run cost on a T4: feature extraction ~25 min (three splits),
probe fit ~2 min, attribute head ~10 min, CPAS-MLP training ~1 GPU-hour
(skipped when the released checkpoint is present). Everything else is seconds.

In [ ]:
# On Colab: mount Drive and unpack the dataset to the runtime's local SSD.
import importlib.util, os
try:
    ON_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:   # no `google` namespace package on local machines
    ON_COLAB = False
if ON_COLAB and not os.path.isdir("/content/datasets/celeba"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/datasets", exist_ok=True)
    # This should take 1-2 minutes; it is deleted when the runtime disconnects.
    os.system("unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/")

In [ ]:
import json
import math
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Subset
from torchvision.datasets import CelebA
from tqdm.auto import tqdm

torch.set_grad_enabled(False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

### Delivered configuration

The hyperparameters below were swept on a **held-out validation slice of the
train split** (§6), never on the 14 test queries. They are hardcoded here with
their provenance because they live in the sweep CSVs, not in the checkpoints.

In [ ]:
# Swept on validation by scripts/run_attribute_retrieval.py
# (results/attribute_retrieval_cpas_soft_sweep.csv): lam = 4, w_cos = 1.
CFG = {
    "soft_reference": True,            # rung-1 knob, see S3
    "lam_constraint": 4.0,             # soft constraint penalty beat the hard filter
    "w_cos": 1.0,                      # weight of the fusion-module cosine term
    "head_path": Path("results/attribute_head.pt"),      # MLP head, full train split
    "cpas_path": Path("results/mlp_final_s0.pt"),        # CPAS-MLP, criterion-aligned miner
    "probe_path": Path("results/probe_weights.pt"),
}
SOURCE_CHUNK = 512     # references scored per block; lower it if RAM is tight
RUN_SWEEP = False      # True re-runs the validation sweep instead of trusting CFG

### Paths and data

`get_paths` resolves Colab vs. local locations. **Indexing warning** (assignment
§3.1.2): ground-truth keys are *dataset indices*, not filenames — index 13 of
the test split is file `182651.jpg`, not `000013.jpg`. Every access below goes
through `celeba[int(key)]`; nothing ever builds a file path from an index.

In [ ]:
REPO_ROOT = Path.cwd()  # the notebook lives at the repo root


def _on_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


@dataclass(frozen=True)
class Paths:
    data_root: Path
    annotations_path: Path
    features_dir: Path


def get_paths() -> Paths:
    if _on_colab():
        return Paths(
            data_root=Path("/content/datasets"),
            annotations_path=Path("/content/drive/MyDrive/datasets/celeba_evaluation.json"),
            features_dir=Path("/content/drive/MyDrive/datasets/features"),
        )
    return Paths(
        data_root=REPO_ROOT,
        annotations_path=REPO_ROOT / "celeba_evaluation.json",
        features_dir=REPO_ROOT / "features",
    )


def load_dataset(paths: Paths, split: str = "test") -> CelebA:
    # Note: CelebA appends "celeba/" to the root itself.
    if not (paths.data_root / "celeba").is_dir():
        raise FileNotFoundError(
            f"CelebA folder not found under {paths.data_root}. "
            "Expected <data_root>/celeba/img_align_celeba/..."
        )
    return CelebA(root=paths.data_root, split=split, download=False)


def load_annotations(paths: Paths) -> list[dict]:
    if not paths.annotations_path.is_file():
        raise FileNotFoundError(f"Evaluation JSON not found: {paths.annotations_path}")
    with open(paths.annotations_path) as f:
        return json.load(f)

In [ ]:
paths = get_paths()
celeba = load_dataset(paths)                    # test split: the retrieval database
annotations = load_annotations(paths)           # the 14 mandatory queries + ground truth
labels = celeba.attr.bool()                     # (N, 40) true attribute codes
attributes = [n for n in celeba.attr_names if n]
attr_index = {name: i for i, name in enumerate(attributes)}
N_IMAGES, N_ATTR = labels.shape
print(f"test split: {N_IMAGES} images, {N_ATTR} attributes, "
      f"{len(annotations)} queries, "
      f"{sum(len(e['ground_truth']) for e in annotations)} (query, reference) pairs")
# The S3.1.2 trap, demonstrated: dataset index 13 is not file 000013.jpg.
print(f"filename at test index 13: {celeba.filename[13]}")

## 1. The criterion the task is defined by

Everything in this notebook follows from one fact, so it comes first.

**Assignment §3.1.1** defines a retrieved image as a correct answer **iff**

1. it strictly satisfies the query's positive/negative constraints, **and**
2. its remaining attributes are within **Hamming distance 2** of the
   reference's.

Both conditions are statements about 40-bit **attribute codes** — not about
embedding geometry. Three consequences drive every design choice below:

- **The composition is not a learning problem.** The optimal target code is the
  reference's code with the queried bits forced ($+a \to 1$, $-a \to 0$).
  §3.1.1 *defines* it that way; there is nothing for a fusion module to
  discover about *what* to aim at — only about how to represent and score it.
- **Retrieval quality is bounded by attribute-prediction accuracy.** With true
  labels the benchmark is solved exactly (R@10 = 1.000, measured in §9).
- **Cosine similarity to a composed vector is a weak proxy.** CLIP similarity
  and attribute-code proximity are different orderings: the cosine top-10
  differs from the reference on 4–10 attributes while correct answers differ
  on 0–2, and those correct answers sit at cosine ranks in the thousands.

The rule lives in one shared module — the validation benchmark (§6) and the
miner (§5) both import it, because the two used to carry *separate*
implementations of "correct answer", and only one of them was right. That
defect and its measured cost are discussed in §5.

In [ ]:
MAX_HAMMING = 2   # assignment S3.1.1 (2)
MIN_TARGETS = 5   # assignment S3.1.1 inclusion rule: ">= 5 valid ground-truth targets"


def satisfies(labels: torch.Tensor, add: list[int], remove: list[int]) -> torch.Tensor:
    """(N,) bool: has every attribute in `add` and none in `remove`.

    labels: (N, A) bool. An empty query is satisfied by everything.
    """
    ok = torch.ones(labels.shape[0], dtype=torch.bool, device=labels.device)
    if add:
        ok &= labels[:, add].all(dim=1)
    if remove:
        ok &= ~labels[:, remove].any(dim=1)
    return ok


def hamming_to(
    labels: torch.Tensor,
    code: torch.Tensor,
    queried: list[int],
) -> torch.Tensor:
    """(N,) int: disagreements with `code` over the NON-queried attributes.

    labels: (N, A) bool; code: (A,) bool; queried: attribute rows the query
    names. Those differ from the reference by construction - that is what the
    query asked for - so S3.1.1 excludes them from the distance.

    Computed as "all disagreements minus the queried ones" rather than by
    slicing the complement: it avoids materialising a second (N, A-|queried|)
    copy on every call, and the miner makes one call per sampled query.
    """
    diff = labels != code
    total = diff.sum(dim=1)
    if not queried:
        return total
    return total - diff[:, queried].sum(dim=1)


def valid_mask(
    labels: torch.Tensor,
    code: torch.Tensor,
    add: list[int],
    remove: list[int],
    max_hamming: int = MAX_HAMMING,
) -> torch.Tensor:
    """(N,) bool: the S3.1.1 correct answers for this reference code and query.

    Does not exclude the reference itself - callers that hold its row index are
    the ones that can, and the batch-level mask in src/training.py works on a
    code without knowing any index.
    """
    queried = list(add) + list(remove)
    return satisfies(labels, add, remove) & (
        hamming_to(labels, code, queried) <= max_hamming
    )

### Verifying the rule against the released ground truth

Before trusting anything downstream we reconstruct the ground-truth sets for
**all (query, reference) pairs** from the labels alone and compare them with
the JSON, exactly. The check uses an independent implementation (two float
matrix products rather than `valid_mask`), so it validates the rule *and* the
whole data path — features aligned to indices, labels from the right split,
no filename trap — in one pass. **If this cell reports any mismatch, nothing
below is meaningful.**

In [ ]:
label_f = labels.float()
# parse_query is defined with the retrieval components in S4; inline the split here.
def _parse(q):
    pos, neg = [], []
    for tok in q.split(","):
        tok = tok.strip()
        (pos if tok.startswith("+") else neg).append(tok[1:].strip())
    return pos, neg

mismatches = checked = 0
for entry in annotations:
    pos, neg = _parse(entry["query"])
    pr = [attr_index[a] for a in pos]; nr = [attr_index[a] for a in neg]
    others = [a for a in range(N_ATTR) if a not in set(pr) | set(nr)]
    ok = satisfies(labels, pr, nr)                                   # condition (1)
    srcs = [int(k) for k in entry["ground_truth"]]
    for start in range(0, len(srcs), SOURCE_CHUNK):
        chunk = srcs[start:start + SOURCE_CHUNK]
        r = label_f[chunk][:, others]
        p = label_f[:, others]
        hamming = p @ (1 - r).T + (1 - p) @ r.T                      # exact on 0/1
        valid = ok.unsqueeze(1) & (hamming <= MAX_HAMMING)           # condition (2)
        for col, src in enumerate(chunk):
            expected = torch.zeros(N_IMAGES, dtype=torch.bool)
            expected[entry["ground_truth"][str(src)]] = True
            rebuilt = valid[:, col].clone(); rebuilt[src] = False
            mismatches += not torch.equal(rebuilt, expected); checked += 1
print(f"reconstructed {checked} pairs, {mismatches} mismatches")
assert mismatches == 0, "S3.1.1 reconstruction failed - do not trust anything below"

## 2. Positioning

Recent work reframes VLM embedding spaces for conditioned retrieval. GDE
(Berasi et al., 2025) decomposes entangled embeddings into separate semantic
structures; CLAY (Lim et al., 2026) makes the embedding space text-conditional,
decoupling conditioning from visual feature extraction. The assignment's stated
limitation of that line is the **naïve pre-SVD stacking** of condition
embeddings: concatenation offers no dynamic control over how multiple,
possibly conflicting conditions combine.

Our answer replaces the stacked-embedding fusion with two components that are
*explicit about what they represent*:

1. **A learned attribute space.** A small head on frozen CLIP features predicts
   all 40 attribute probabilities per image (§4). This is the projection in
   which conditions stop being entangled: each condition is an axis.
2. **A fusion rule that is exact in that space.** The composite query is the
   reference's predicted code with the queried bits forced — the provably
   optimal target under §3.1.1 — and the similarity metric scores it directly
   (§3). Positive and negative constraints enter through different terms with
   different signs; this is our answer to the assignment's requirement to
   *"define how these positive and negative constraints interact in the
   embedding space."*

A trained fusion module in the *original* CLIP space, **CPAS-MLP** (§4.4), is
retained and enters the final score through a cosine term. Following CLAY, the
visual database is embedded **once, offline, and kept frozen**; only the query
side is computed at retrieval time.

## 3. The score: ranking by the criterion itself

For a database image $d$ with predicted attribute probabilities
$p(d) \in [0,1]^{40}$ and thresholded code $c(d)$, a reference with target code
$r$ (§3.1.1-forced), and a composite query embedding $q$ from the fusion
module:

$$
\text{score}(d) \;=\;
\underbrace{-\sum_{a \notin \text{query}} \big[\, p_a(d)\,(1-r_a) + (1-p_a(d))\,r_a \,\big]}_{\text{expected Hamming — condition (2)}}
\;\;\underbrace{-\;\lambda\,\big[\, c(d) \text{ breaks a queried constraint} \,\big]}_{\text{condition (1)}}
\;\;\underbrace{+\;w \, (q \cdot d)}_{\text{fusion module}}
$$

Sort descending, take the top-$k$. Three design points:

**The first term is itself an inner product — the composite query in attribute
space.** Expanding, $\mathbb{E}[\text{Hamming}] = \sum_a p_a(1-2r_a) + \sum_a r_a$;
the second sum is candidate-independent, so with $s_a = 2r_a - 1 \in \{-1,+1\}$,

$$ \text{ranking by } -\mathbb{E}[\text{Hamming}] \;\equiv\; \text{ranking by } \langle s,\, p(d) \rangle. $$

The method *is* cosine-style retrieval — taken in $\mathbb{R}^{40}$, where the
criterion's coordinates are the axes, instead of in $\mathbb{R}^{512}$, where
CLIP's pretraining fixed the geometry. §8 measures what this difference buys.

**The soft reference.** The reference's code is itself predicted. Thresholding
it first spends the Hamming budget on noise: a reference at $p = 0.49$ and a
candidate at $p = 0.52$ — two coin flips — would register a *full*
disagreement, half of the ±2 budget. Passing the reference's probabilities
instead makes each non-queried attribute weigh exactly the predictor's
confidence in it, and an attribute at $p = 0.5$ becomes unable to reorder
candidates at all. Queried bits are still forced to exact 0/1 — the *query* is
certain even when the predictor is not. With hard 0/1 inputs the soft form
reproduces the thresholded computation bit for bit, so it is a strict
generalisation (ablated in §7).

**The constraint term is non-compensatory, and deliberately soft.** A single
inner product is compensatory — a surplus on one attribute pays for a violation
on another — while §3.1.1's condition (1) is conjunctive. The indicator
restores that. But $\lambda$ stays finite and *small*: the penalty acts on
**predicted** codes, and a hard filter ($\lambda \to \infty$) propagates
prediction errors irreversibly. On validation $\lambda = 4$ matched or beat
$\lambda \in \{16, 100\}$ (§10 discusses the saturation). The database-side
code stays thresholded here: a yes/no condition needs a yes/no code.

The hinge-penalty module below is the constraint term's ancestor (used
standalone on the cosine pipeline); the attribute-space module implements the
full score.

In [ ]:
def database_probe_probs(
    features: torch.Tensor,
    weights: torch.Tensor,
    biases: torch.Tensor,
) -> torch.Tensor:
    """(N, A) probability that each database image has each attribute.

    features: (N, D) L2-normalized image features.
    weights:  (A, D) RAW probe weights - not the normalized directions that the
        composition uses. `src.probes.load_probes` normalizes its weights while
        the saved biases belong to the unnormalized ones, so pairing the two
        gives plausible-looking but meaningless probabilities; use
        `src.probes.load_raw_probes` here.
    biases:   (A,)
    """
    if weights.shape[0] != biases.shape[0]:
        raise ValueError("weights and biases must describe the same attributes")
    return torch.sigmoid(features @ weights.T + biases)


def exclusion_penalty(
    db_probs: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
    lam_neg: float = 0.0,
    lam_pos: float = 0.0,
    thresholds: torch.Tensor | None = None,
    hinge: bool = True,
) -> torch.Tensor:
    """(N,) non-negative penalty per database image for one query's constraints.

    thresholds: (A,) per-attribute tau, or None for a shared 0.5.
    hinge=False replaces relu(p - tau) with p (and relu(tau - p) with 1 - p),
    which is ablation row 4: a linear penalty is still compensatory - it is just
    another term in a weighted sum, expressible by moving `q` itself.
    """
    n, a = db_probs.shape
    tau = (
        torch.full((a,), 0.5, dtype=db_probs.dtype, device=db_probs.device)
        if thresholds is None
        else thresholds.to(db_probs)
    )
    penalty = torch.zeros(n, dtype=db_probs.dtype, device=db_probs.device)
    if lam_neg and neg_rows:
        excess = db_probs[:, neg_rows] - (tau[neg_rows] if hinge else 0.0)
        penalty += lam_neg * (excess.relu() if hinge else excess).sum(dim=1)
    if lam_pos and pos_rows:
        deficit = (tau[pos_rows] if hinge else 1.0) - db_probs[:, pos_rows]
        penalty += lam_pos * (deficit.relu() if hinge else deficit).sum(dim=1)
    return penalty


@dataclass(frozen=True)
class Rerank:
    """Scoring configuration shared by every call site.

    db_probs: (N, A) from `database_probe_probs`, computed once for the whole
        database (19,962 x 40 floats, ~3 MB) - scoring a query is then one
        gather and one hinge.
    top_m: apply the penalty to the top-m candidates by cosine only (ablation
        row 5, the cheap two-stage version); None penalizes the whole database.
    """

    db_probs: torch.Tensor
    lam_neg: float = 0.0
    lam_pos: float = 0.0
    thresholds: torch.Tensor | None = None
    hinge: bool = True
    top_m: int | None = None

    @property
    def active(self) -> bool:
        return bool(self.lam_neg) or bool(self.lam_pos)

    def scores(
        self,
        query_vecs: torch.Tensor,
        image_features: torch.Tensor,
        pos_rows: list[int],
        neg_rows: list[int],
    ) -> torch.Tensor:
        """(S, N) re-ranked scores, higher is better."""
        sims = query_vecs @ image_features.T
        if not self.active:
            return sims
        penalty = exclusion_penalty(
            self.db_probs.to(sims.device), pos_rows, neg_rows,
            self.lam_neg, self.lam_pos, self.thresholds, self.hinge,
        )
        if self.top_m is None:
            return sims - penalty
        # Two-stage: only the shortlist is charged, so candidates below it can
        # no longer be displaced by a penalized shortlist member.
        m = min(self.top_m, sims.shape[1])
        shortlist = sims.topk(m, dim=1).indices
        charged = sims.clone()
        charged.scatter_add_(1, shortlist, -penalty[shortlist])
        return charged


def rank_with_exclusion(
    query_vecs: torch.Tensor,
    image_features: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
    rerank: Rerank | None = None,
    exclude: list[int] | None = None,
) -> torch.Tensor:
    """Image indices sorted by descending penalized score.

    Same contract as `src.retrieval.rank` (including `exclude` semantics), so a
    benchmark loop switches with one line. With `rerank=None` or an inactive
    configuration this is exactly `rank`.
    """
    if exclude is not None and len(exclude) != query_vecs.shape[0]:
        raise ValueError(
            f"exclude must have one entry per query row: "
            f"got {len(exclude)} for {query_vecs.shape[0]} rows"
        )
    if rerank is None:
        scores = query_vecs @ image_features.T
    else:
        scores = rerank.scores(query_vecs, image_features, pos_rows, neg_rows)
    if exclude is not None:
        rows = torch.arange(scores.shape[0], device=scores.device)
        scores[rows, torch.tensor(exclude, device=scores.device)] = float("-inf")
    return scores.argsort(dim=1, descending=True)

In [ ]:
def target_code(
    ref_code: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
) -> torch.Tensor:
    """The reference's code with the queried bits forced to the query.

    (R, A) bool in, (R, A) bool out. Nothing is learned here: under S3.1.1 the
    optimal target code is exactly the reference's code with the queried bits
    flipped, so the composition is determined by the ground-truth rule.
    """
    code = ref_code.clone()
    if pos_rows:
        code[:, pos_rows] = True
    if neg_rows:
        code[:, neg_rows] = False
    return code


def expected_hamming(
    db_probs: torch.Tensor,
    ref_code: torch.Tensor,
    rows: list[int],
    weights: torch.Tensor | None = None,
) -> torch.Tensor:
    """(N, R) expected number of attributes in `rows` where a database image
    disagrees with each reference code.

    E[disagreements] = sum_a  w_a [ p_a (1 - r_a) + (1 - p_a) r_a ], computed as
    two matrix products so the whole database is scored for a block of
    references at once.

    db_probs: (N, A) predicted probabilities; ref_code: (R, A) bool, or float
    probabilities to keep the reference's own uncertainty in the distance -
    thresholding it first spends the Hamming budget on attributes neither side
    knew anything about.

    weights: optional (A,) per-attribute weight, indexed by the same `rows`.
    None means a uniform weight of 1 and reproduces the unweighted distance
    exactly, which is what makes the weighting separately ablatable.
    """
    if not rows:
        return torch.zeros(db_probs.shape[0], ref_code.shape[0],
                           device=db_probs.device)
    p = db_probs[:, rows]
    r = ref_code[:, rows].to(p.dtype)
    if weights is None:
        return p @ (1 - r).T + (1 - p) @ r.T
    # w - pw is w * (1 - p): scaling the candidate side keeps both matmuls.
    w = weights[rows].to(p)
    pw = p * w
    return pw @ (1 - r).T + (w - pw) @ r.T


def constraint_violation(
    db_code: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
) -> torch.Tensor:
    """(N,) 1.0 where a database image breaks any queried constraint, else 0."""
    bad = torch.zeros(db_code.shape[0], dtype=torch.bool, device=db_code.device)
    if pos_rows:
        bad |= ~db_code[:, pos_rows].all(dim=1)
    if neg_rows:
        bad |= db_code[:, neg_rows].any(dim=1)
    return bad.to(torch.float32)


def attribute_scores(
    db_probs: torch.Tensor,
    db_code: torch.Tensor,
    ref_code: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
    lam_constraint: float = 100.0,
    cosine: torch.Tensor | None = None,
    w_cos: float = 0.0,
    weights: torch.Tensor | None = None,
) -> torch.Tensor:
    """(R, N) scores, higher is better, for one query and a block of references.

    db_probs / db_code: (N, A) predicted probabilities and thresholded code;
    ref_code: (R, A) bool codes of the references (already query-adjusted by
    `target_code`, or raw - the queried columns are excluded either way);
    cosine: optional (N, R) similarity of the composite query embedding to the
    database, blended in with weight w_cos;
    weights: optional (A,) per-attribute weight for the Hamming term. The
    constraint term is deliberately left unweighted - it is a conjunctive
    condition, not a distance.
    """
    others = [a for a in range(db_probs.shape[1])
              if a not in set(pos_rows) | set(neg_rows)]
    score = -expected_hamming(db_probs, ref_code, others, weights)  # (N, R)
    score = score - lam_constraint * constraint_violation(
        db_code, pos_rows, neg_rows
    ).unsqueeze(1)
    if cosine is not None and w_cos:
        score = score + w_cos * cosine
    return score.T


def rank_by_attributes(
    db_probs: torch.Tensor,
    db_code: torch.Tensor,
    ref_code: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
    exclude: list[int] | None = None,
    **kwargs,
) -> torch.Tensor:
    """Database indices sorted by descending attribute score, one row per reference.

    Same contract as `src.retrieval.rank`, including `exclude` semantics, so it
    is a drop-in replacement in a benchmark loop.
    """
    scores = attribute_scores(db_probs, db_code, ref_code, pos_rows, neg_rows,
                              **kwargs)
    if exclude is not None:
        if len(exclude) != scores.shape[0]:
            raise ValueError(
                f"exclude must have one entry per reference: got {len(exclude)} "
                f"for {scores.shape[0]} rows"
            )
        rows = torch.arange(scores.shape[0], device=scores.device)
        scores[rows, torch.tensor(exclude, device=scores.device)] = float("-inf")
    return scores.argsort(dim=1, descending=True)

## 4. Architecture

```
CelebA image ──► [4.1] frozen CLIP ──► features (512-d, L2-normalized, cached)
                                          │
                                          ├──► [4.2-4.3] attribute predictors ──► p(d) ∈ [0,1]^40
                                          │
      T+ / T− ──────────────────────────────────────────────► target code r
                                          │
                                          └──► [4.4] CPAS-MLP fusion Φ ──► q
                                                              │
                              score(d) = −E[Hamming] − λ·violation + w·(q·d)
```

### 4.1 Frozen features

CLIP ViT-B/32 (`openai/clip-vit-base-patch32`, as required by §3.3), image
encoder only, L2-normalized, extracted once per split and cached. Nothing in
this project ever fine-tunes the encoder.

In [ ]:
class ClipEncoder:
    MODEL_NAME = "openai/clip-vit-base-patch32"

    def __init__(self, device: str | None = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self._model = None
        self._processor = None

    def _load(self):
        if self._model is None:
            from transformers import CLIPModel, CLIPProcessor

            self._model = CLIPModel.from_pretrained(self.MODEL_NAME).to(self.device).eval()
            self._processor = CLIPProcessor.from_pretrained(self.MODEL_NAME)
        return self._model, self._processor

    @torch.no_grad()
    def encode_texts(self, prompts: list[str]) -> torch.Tensor:
        model, processor = self._load()
        inputs = processor(
            text=prompts, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        feats = model.get_text_features(**inputs).pooler_output.cpu().float()
        return feats / feats.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def encode_images(self, dataset, batch_size: int = 64,
                      limit: int | None = None) -> torch.Tensor:
        model, processor = self._load()
        n = len(dataset) if limit is None else min(limit, len(dataset))
        chunks = []
        for start in tqdm(range(0, n, batch_size), desc="Encoding images"):
            batch = [dataset[i][0] for i in range(start, min(start + batch_size, n))]
            inputs = processor(images=batch, return_tensors="pt").to(self.device)
            feats = model.get_image_features(**inputs).pooler_output.cpu().float()
            chunks.append(feats)
        feats = torch.cat(chunks)
        return feats / feats.norm(dim=-1, keepdim=True)


def resolve_pool(features_dir, override=None) -> Path:
    """Path to the train-split feature pool: the full split if it exists.

    Every script that trains on train-split features resolves it the same way,
    so extracting the full split with `extract_train_features.py --all` upgrades
    all of them at once instead of only the ones that happened to look for it.
    """
    if override is not None:
        return Path(override)
    slug = ClipEncoder.MODEL_NAME.split("/")[-1]
    full = Path(features_dir) / f"{slug}_train.pt"
    return full if full.is_file() else Path(features_dir) / f"{slug}_train30k.pt"


def load_pool(path) -> tuple[torch.Tensor, torch.Tensor]:
    """(features, train-split indices) from either cache layout.

    The sampled pool stores {"features", "indices"}; the full split is a plain
    tensor in dataset order, so its indices are simply 0..N-1. Callers need the
    indices to align train-split labels with the rows.
    """
    saved = torch.load(path, weights_only=True)
    if isinstance(saved, dict):
        return saved["features"], saved["indices"]
    return saved, torch.arange(saved.shape[0])


def cache_path(features_dir: Path, split: str) -> Path:
    model_slug = ClipEncoder.MODEL_NAME.split("/")[-1]
    return features_dir / f"{model_slug}_{split}.pt"


def load_or_extract(encoder, dataset, features_dir: Path, split: str = "test") -> torch.Tensor:
    path = cache_path(features_dir, split)
    if path.is_file():
        return torch.load(path, weights_only=True)
    features = encoder.encode_images(dataset)
    features_dir.mkdir(parents=True, exist_ok=True)
    torch.save(features, path)
    return features

Query parsing and the zero-shot composition (`compose_text` here;
`compose` in the repo's `src/retrieval.py`) — the assignment's required
baseline: $q = \mathrm{normalize}(\gamma\, v_{ref} + \sum t^+ - \sum t^-)$ over
CLIP *text* embeddings of hand-written prompts.

In [ ]:
# Prompt templates in the style of the original CLIP paper (Radford et al., 2021).
# One entry per attribute appearing in the 14 benchmark queries; extend as needed.
PROMPTS: dict[str, str] = {
    "Smiling": "a photo of a smiling person",
    "Eyeglasses": "a photo of a person wearing eyeglasses",
    "Heavy_Makeup": "a photo of a person with heavy makeup",
    "Male": "a photo of a man",
    "Young": "a photo of a young person",
    "Blond_Hair": "a photo of a person with blond hair",
    "Mustache": "a photo of a person with a mustache",
    "Black_Hair": "a photo of a person with black hair",
    "Wavy_Hair": "a photo of a person with wavy hair",
    "Chubby": "a photo of a chubby person",
    "Wearing_Hat": "a photo of a person wearing a hat",
    "Wearing_Lipstick": "a photo of a person wearing lipstick",
}


def parse_query(query: str) -> tuple[list[str], list[str]]:
    """Parse '+A, -B' into (positives, negatives) lists of attribute names."""
    positives: list[str] = []
    negatives: list[str] = []
    for token in query.split(","):
        token = token.strip()
        if token.startswith("+"):
            positives.append(token[1:].strip())
        elif token.startswith("-"):
            negatives.append(token[1:].strip())
        else:
            raise ValueError(f"Query token must start with '+' or '-': {token!r}")
    return positives, negatives


def compose_text(
    v_ref: torch.Tensor,
    pos_texts: torch.Tensor,
    neg_texts: torch.Tensor,
    gamma: float = 1.0,
) -> torch.Tensor:
    """Naive latent arithmetic: normalize(gamma * v_ref + sum(pos) - sum(neg)).

    All inputs are expected L2-normalized. gamma weights identity
    preservation against the attribute edits; gamma=1 is the vanilla
    baseline composition.
    """
    q = gamma * v_ref + pos_texts.sum(dim=0) - neg_texts.sum(dim=0)
    return q / q.norm()


def rank(
    query_vecs: torch.Tensor,
    image_features: torch.Tensor,
    exclude: list[int] | None = None,
) -> torch.Tensor:
    """Return image indices sorted by descending cosine similarity.

    query_vecs: (S, D) normalized query vectors (one row per source image).
    image_features: (N, D) normalized image features.
    exclude: optional list of length S; exclude[i] (the source image of
        row i) is forced to the bottom of row i's ranking. Must have length S.
    """
    if exclude is not None and len(exclude) != query_vecs.shape[0]:
        raise ValueError(
            f"exclude must have one entry per query row: "
            f"got {len(exclude)} for {query_vecs.shape[0]} rows"
        )
    sims = query_vecs @ image_features.T  # (S, N)
    if exclude is not None:
        rows = torch.arange(sims.shape[0])
        sims[rows, torch.tensor(exclude)] = float("-inf")
    return sims.argsort(dim=1, descending=True)

In [ ]:
encoder = ClipEncoder()
features = load_or_extract(encoder, celeba, paths.features_dir)   # (N, 512), cached
print(f"test features: {tuple(features.shape)}")

The training pool: full train split (162,770 images) plus the valid split,
which is the selection surface for the attribute head and its thresholds.
Extraction is skipped when the caches exist.

In [ ]:
slug = ClipEncoder.MODEL_NAME.split("/")[-1]

def extract_split(split: str, out_path: Path) -> None:
    if out_path.is_file():
        return
    ds = load_dataset(paths, split=split)
    feats = encoder.encode_images(ds, batch_size=256)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"features": feats, "indices": torch.arange(len(ds))}, out_path)

pool_path = paths.features_dir / f"{slug}_train.pt"
valid_path = paths.features_dir / f"{slug}_valid.pt"
extract_split("train", pool_path)
extract_split("valid", valid_path)

pool_features, pool_indices = load_pool(pool_path)
train_split = load_dataset(paths, split="train")
pool_labels = train_split.attr[pool_indices].bool()
v = torch.load(valid_path, weights_only=True)
valid_features = v["features"] if isinstance(v, dict) else v
valid_labels = load_dataset(paths, split="valid").attr.bool()
print(f"train pool {tuple(pool_features.shape)}   valid {tuple(valid_features.shape)}")

### 4.2 Linear probes — classifiers *and* edit directions

One logistic regression per attribute on the cached train features, all 40 fit
jointly as a single (40, 512) matrix. A probe plays two roles, and the second
is why it exists here at all:

- **classifier**: $p_a(v) = \sigma(w_a \cdot v + b_a)$ — a calibrated
  probability (raw weights + biases);
- **edit direction**: $\hat{w}_a = w_a / \lVert w_a \rVert$ — the normal of the
  separating hyperplane, i.e. the in-space step that most increases the probe's
  confidence. Because ranking is also a dot product, adding $\hat{w}_a$ to a
  query shifts every database image's score by exactly its probe logit: the
  composition and the classifier perform the same operation. These directions
  are what the fusion module bends (§4.4). MLP weights have no such reading,
  which is why the linear probes are kept alongside the stronger MLP head.

The two loaders below are deliberately distinct: the saved biases belong to the
**raw** weights, so pairing them with normalized directions produces
plausible-looking but meaningless probabilities.

In [ ]:
PROBE_FILE = "probe_weights.pt"


def load_raw_probes(repo_root: Path) -> tuple[torch.Tensor, torch.Tensor, list[str]]:
    """Load saved probes as (raw weights, biases, attribute names).

    Looks in results/ first, then features/ (the weights are a training
    artifact, so either location is valid depending on how they were produced).

    These are the weights the saved biases belong to, so this is what predicts
    a calibrated probability sigmoid(w.d + b) - see src/rerank.py. The
    composition wants `load_probes` instead.
    """
    for folder in ("results", "features"):
        path = repo_root / folder / PROBE_FILE
        if path.is_file():
            saved = torch.load(path, map_location="cpu", weights_only=True)
            return saved["weights"], saved["biases"], saved["attributes"]
    raise FileNotFoundError(
        f"{PROBE_FILE} not found in {repo_root}/results or {repo_root}/features; "
        "run scripts/fit_probes.py first."
    )


def load_probes(repo_root: Path) -> tuple[torch.Tensor, torch.Tensor, list[str]]:
    """Load saved probes as (normalized directions, biases, attribute names).

    The directions are L2-normalized but the biases are not rescaled with them:
    the biases belong to the raw weights, and only the directions are used by
    the composition. Anything that needs sigmoid(w.d + b) must use
    `load_raw_probes`.
    """
    w, biases, attributes = load_raw_probes(repo_root)
    return w / w.norm(dim=1, keepdim=True), biases, attributes


def fit_linear_probes(
    features: torch.Tensor,
    labels: torch.Tensor,
    epochs: int = 2000,
    lr: float = 0.05,
    weight_decay: float = 0.0,
    class_balanced: bool = False,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Fit A independent logistic regressions with full-batch Adam.

    The defaults are the recipe that produced results/probe_weights.pt and
    every number in results/ downstream of it, so they must not change:
    changing them silently invalidates the reported benchmarks. At those
    defaults the probes reach a macro-mean valid AUC of 0.929, from 0.731
    (Oval_Face) to 0.999 (Male). The two optional knobs allow a refit under
    regularization without disturbing the default recipe.

    weight_decay is Adam's coupled L2 (added to the gradient), not AdamW's
    decoupled form. class_balanced sets pos_weight = n_neg / n_pos per
    attribute, so a rare attribute's positives are not drowned by its
    negatives; an attribute with no positives gets weight 1, since dividing
    by zero there would poison every other attribute's gradient.

    features: (N, D) L2-normalized; labels: (N, A) 0/1.
    Returns weights (A, D) and biases (A,), detached.
    """
    n, d = features.shape
    a = labels.shape[1]
    # Follow the features: on the full train split this is a GPU tensor, and
    # zero-init on CPU would fail in the first matmul.
    device = features.device
    w = torch.zeros(a, d, device=device, requires_grad=True)
    b = torch.zeros(a, device=device, requires_grad=True)
    y = labels.float().to(device)

    pos_weight = None
    if class_balanced:
        n_pos = y.sum(dim=0)
        pos_weight = torch.where(n_pos > 0, (n - n_pos) / n_pos.clamp(min=1.0),
                                 torch.ones_like(n_pos))

    opt = torch.optim.Adam([w, b], lr=lr, weight_decay=weight_decay)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(features @ w.T + b, y)
        loss.backward()
        opt.step()
    return w.detach(), b.detach()


def compose_probe(
    v_ref: torch.Tensor,
    pos_dirs: torch.Tensor,
    neg_dirs: torch.Tensor,
    gamma: float = 1.0,
) -> torch.Tensor:
    """Composite query embedding from probe directions.

    q = normalize(gamma * v_ref + sum(pos_dirs) - sum(neg_dirs))

    pos_dirs / neg_dirs: (K, D) L2-normalized probe weight directions for the
    query's T+ / T- attributes. Same shape contract as retrieval.compose,
    with the text embeddings swapped for in-space attribute directions and an
    explicit identity weight gamma.
    """
    q = gamma * v_ref + pos_dirs.sum(dim=0) - neg_dirs.sum(dim=0)
    return q / q.norm()


def roc_auc(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """Threshold-free ranking quality (Mann-Whitney U / rank-sum form)."""
    order = scores.argsort()
    ranks = torch.empty(len(scores), dtype=torch.float64)
    ranks[order] = torch.arange(1, len(scores) + 1, dtype=torch.float64)
    pos = labels.bool()
    n_pos = int(pos.sum())
    n_neg = len(labels) - n_pos
    u = ranks[pos].sum().item() - n_pos * (n_pos + 1) / 2
    return u / (n_pos * n_neg)


def average_precision(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """Area under the precision-recall curve, by the standard step-wise sum.

    AP = (1 / P) * sum over positives of precision-at-that-rank. Returns nan
    when `labels` has no positives, which is the only degenerate case.
    """
    n_pos = int(labels.sum())
    if n_pos == 0:
        return float("nan")
    order = scores.argsort(descending=True)
    y = labels[order].to(torch.float64)
    ranks = torch.arange(1, len(y) + 1, dtype=torch.float64)
    precision_at_rank = y.cumsum(0) / ranks
    return float((precision_at_rank * y).sum() / n_pos)


def score_attributes(
    scores: torch.Tensor,
    labels: torch.Tensor,
) -> tuple[list[float], list[float]]:
    """Per-attribute (AUC, AP) from an (N, A) score matrix and (N, A) labels."""
    aucs, aps = [], []
    for j in range(labels.shape[1]):
        aucs.append(roc_auc(scores[:, j], labels[:, j]))
        aps.append(average_precision(scores[:, j], labels[:, j]))
    return aucs, aps

In [ ]:
if not CFG["probe_path"].is_file():
    print("fitting linear probes on the train pool (deterministic recipe)...")
    Wp, Bp = fit_linear_probes(pool_features.to(DEVICE), pool_labels.to(DEVICE))
    CFG["probe_path"].parent.mkdir(exist_ok=True)
    torch.save({"weights": Wp.cpu(), "biases": Bp.cpu(), "attributes": attributes},
               CFG["probe_path"])
directions, _, probe_attributes = load_probes(Path.cwd())
W, B, _ = load_raw_probes(Path.cwd())
assert probe_attributes == attributes, "probe/label attribute order mismatch"
probe_logits_valid = valid_features @ W.T + B
aucs, aps = score_attributes(probe_logits_valid, valid_labels.float())
print(f"probes {tuple(W.shape)}: valid macro AUC {sum(aucs) / N_ATTR:.3f}, "
      f"macro AP {sum(aps) / N_ATTR:.3f}")

### 4.3 The MLP attribute head — where the performance is

The score consumes codes, so **per-bit accuracy is the quantity to optimize**
— not AUC, which measures ranking *within* an attribute and flatters rare ones.
Correctness requires landing inside a radius-2 ball across ~38 bits, and errors
compound: at 0.909 per-bit accuracy the expected code is already ~3.5 bits
wrong. Measured exchange rate on this benchmark: **+0.003 bit accuracy bought
+0.022 R@10**, roughly 7× amplification.

Two details that matter:

- **Selection on held-out bit accuracy, not loss** — accuracy is what transfers
  to the score.
- **Per-attribute decision thresholds**, tuned on the valid split and saved
  *with* the weights: 0.5 is only optimal for a calibrated predictor on a
  balanced attribute, and most CelebA attributes are far from balanced. A code
  produced with different thresholds is a different code.

In [ ]:
class AttributeMLP(nn.Module):
    """Two hidden layers on frozen CLIP features, one logit per attribute.

    Deliberately small: the features are frozen and the training signal is 40
    binary labels, so capacity is not the constraint. Widths of 512-2048 all
    land within 0.001 bit accuracy of each other.
    """

    def __init__(self, d: int = 512, hidden: int = 1024, attributes: int = 40,
                 dropout: float = 0.2) -> None:
        super().__init__()
        self.config = {"d": d, "hidden": hidden, "attributes": attributes,
                       "dropout": dropout}
        self.net = nn.Sequential(
            nn.Linear(d, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, attributes),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """(N, D) features -> (N, A) logits."""
        return self.net(features)


@torch.no_grad()
def bit_accuracy(logits: torch.Tensor, labels: torch.Tensor,
                 thresholds: torch.Tensor | None = None) -> float:
    """Fraction of individual attribute bits predicted correctly."""
    probs = torch.sigmoid(logits)
    pred = probs > (0.5 if thresholds is None else thresholds)
    return float((pred == labels.bool()).float().mean())


@torch.no_grad()
def tune_thresholds(logits: torch.Tensor, labels: torch.Tensor,
                    steps: int = 91) -> torch.Tensor:
    """Per-attribute decision threshold maximizing bit accuracy on held-out data.

    0.5 is only optimal for a calibrated probe on a balanced attribute, and most
    CelebA attributes are far from balanced. Must be fit on data the predictor
    was not trained on, or it overfits the training split's base rates.

    Returns (A,) thresholds.
    """
    probs = torch.sigmoid(logits)
    grid = torch.linspace(0.05, 0.95, steps)
    out = torch.empty(probs.shape[1])
    for j in range(probs.shape[1]):
        acc = torch.tensor([
            float(((probs[:, j] > t) == labels[:, j].bool()).float().mean())
            for t in grid
        ])
        out[j] = grid[int(acc.argmax())]
    return out


@torch.no_grad()
def attribute_reliability(pred: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """(A,) Youden's J per attribute: sensitivity + specificity - 1.

    Zero for a predictor that answers one class regardless of its input, one for
    a perfect one. Per-attribute *accuracy* cannot serve here: CelebA attributes
    are heavily imbalanced, so always answering the majority class scores 88% on
    Wearing_Necklace (12% positive rate) while detecting nothing at all, and
    would earn a large weight for an attribute the predictor is blind to.

    pred / labels: (N, A) bool, thresholded predictions and true labels, so the
    same function serves the linear probe and the MLP head. An attribute with no
    positives or no negatives in `labels` returns 0: its rate is undefined, and
    0 is exactly the "carries no information" weight.
    """
    p, y = pred.bool(), labels.bool()
    pos = y.sum(0).float()
    neg = (~y).sum(0).float()
    sensitivity = (p & y).sum(0).float() / pos.clamp(min=1.0)
    specificity = ((~p) & (~y)).sum(0).float() / neg.clamp(min=1.0)
    j = sensitivity + specificity - 1.0
    return torch.where((pos > 0) & (neg > 0), j, torch.zeros_like(j))


@torch.no_grad()
def reliability_weights(pred: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """(A,) non-negative per-attribute weights averaging 1, from held-out data.

    Youden's J clamped at zero and rescaled. The clamp is not cosmetic: a
    negative weight would invert the target bit, so the score would actively ask
    for the opposite of what the reference has, and on held-out data a negative
    J is noise rather than an anti-correlated attribute worth exploiting.

    The rescaling keeps the Hamming term on the scale lam_constraint was swept
    against, so a run with weights stays comparable to one without.
    """
    j = attribute_reliability(pred, labels).clamp(min=0.0)
    mean = j.mean()
    return j / mean if float(mean) > 0 else torch.ones_like(j)


def fit_attribute_head(
    features: torch.Tensor,
    labels: torch.Tensor,
    val_features: torch.Tensor,
    val_labels: torch.Tensor,
    hidden: int = 1024,
    dropout: float = 0.2,
    epochs: int = 60,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 512,
    device: str = "cpu",
    log: bool = True,
) -> tuple[AttributeMLP, float]:
    """Train an AttributeMLP, keeping the epoch with the best val bit accuracy.

    Selection is on held-out bit accuracy rather than loss: the retrieval score
    consumes the thresholded code, so accuracy is the quantity that transfers.

    Returns (model with the best weights loaded, that val bit accuracy).
    """
    model = AttributeMLP(d=features.shape[1], hidden=hidden,
                         attributes=labels.shape[1], dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    loss_fn = nn.BCEWithLogitsLoss()
    y = labels.float().to(device)
    x = features.to(device)
    vx, vy = val_features.to(device), val_labels.to(device)

    best, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(x.shape[0], device=device)
        for start in range(0, x.shape[0], batch_size):
            rows = perm[start : start + batch_size]
            opt.zero_grad()
            loss_fn(model(x[rows]), y[rows]).backward()
            opt.step()
        sched.step()
        model.eval()
        acc = bit_accuracy(model(vx), vy)
        if acc > best:
            best = acc
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        if log and (epoch + 1) % 10 == 0:
            print(f"  epoch {epoch + 1:3d}  val bit accuracy {acc:.4f} "
                  f"(best {best:.4f})", flush=True)
    model.load_state_dict(best_state)
    return model, best


def load_attribute_head(path, device: str = "cpu") -> tuple[AttributeMLP, dict]:
    """Load a head saved by scripts/fit_attribute_head.py."""
    saved = torch.load(path, map_location=device, weights_only=True)
    model = AttributeMLP(**saved["config"]).to(device)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return model, saved

In [ ]:
if not CFG["head_path"].is_file():
    print("training the attribute head on the full train pool...")
    head_model, acc = fit_attribute_head(
        pool_features, pool_labels, valid_features, valid_labels, device=DEVICE)
    val_logits = head_model(valid_features.to(DEVICE)).cpu()
    thresholds = tune_thresholds(val_logits, valid_labels)
    CFG["head_path"].parent.mkdir(exist_ok=True)
    torch.save({"state_dict": {k: t.cpu() for k, t in head_model.state_dict().items()},
                "config": head_model.config, "thresholds": thresholds,
                "attributes": attributes, "val_bit_accuracy": acc,
                "val_bit_accuracy_tuned": bit_accuracy(val_logits, valid_labels, thresholds),
                "pool": pool_path.name, "seed": 0}, CFG["head_path"])
head, head_saved = load_attribute_head(CFG["head_path"])
head_thresholds = head_saved["thresholds"]

probe_acc = bit_accuracy(probe_logits_valid, valid_labels)
print(f"valid bit accuracy   linear probe {probe_acc:.4f}   "
      f"MLP head {head_saved['val_bit_accuracy_tuned']:.4f}   "
      f"(trained on {head_saved.get('pool', '?')})")

**Encoder saturation, measured.** On a 30k train sample every head
configuration (widths 512–2048, dropout 0.2–0.4) plateaued at 0.9119 tuned bit
accuracy. Training on the **full** split — 5.4× the data — moved it to 0.9149:
three thousandths. The plateau is therefore not data-limited; **frozen ViT-B/32
features are saturated for this task**, which §9 turns into a ceiling
statement. Capacity was never the constraint.

### 4.4 The fusion module Φ: CPAS-MLP

The assignment requires a fusion module producing a composite query embedding.
Ours is **CPAS** (Conditioned Per-Attribute Steering): keep the composition
formula of the probe baseline, make its three fixed choices *learned functions
of the query*,

$$ q = \mathrm{normalize}\Big( \gamma(x)\, v_{ref} \;+\; \sum_a s_a \,\alpha_a(x)\, \mathrm{normalize}\big(\hat{w}_a + \Delta_a(x)\big) \Big) $$

with $s_a = \pm 1$ for $T^+/T^-$ and $x$ = (reference, queried set):
$\gamma(x)$ adapts the reference weight, $\alpha_a(x)$ the per-attribute step,
and the bounded bend $\Delta_a(x)$ addresses the documented weakness of fixed
directions — probe normals are **not orthogonal** (blond↔gender, beard↔age, …)
yet the fixed rule adds them as if independent.

The trunk is a per-attribute MLP with a pooled cross-attribute context (an
earlier transformer trunk was ablated away: its cross-attention earned 0.002
against a 0.019 seed spread, while the $\Delta$ head earned the entire gain).
Two properties keep the ablations exact: the composition formula lives in one
shared function, and the heads are initialized so the **untrained module
reproduces the fixed $\gamma = 0.6$ rule bit for bit** — so any gain is
attributable to learned conditioning, not a changed formula.

In [ ]:
def compose(
    v_ref: torch.Tensor,
    dirs: torch.Tensor,
    signs: torch.Tensor,
    gamma: torch.Tensor,
    alpha: torch.Tensor,
    delta: torch.Tensor,
) -> torch.Tensor:
    """Build query embeddings from predicted steering quantities.

    q = normalize( gamma * v_ref + sum_a sign_a * alpha_a * normalize(w_a + delta_a) )

    v_ref: (B, D); dirs/delta: (B, K, D); signs/alpha: (B, K); gamma: (B,).
    alpha and delta are expected to be zeroed on padded slots, which is what
    keeps padding out of the sum. Returns (B, D), L2-normalized.
    """
    bent = nn.functional.normalize(dirs + delta, dim=-1)
    steps = (signs * alpha).unsqueeze(-1) * bent  # (B, K, D)
    q = gamma.unsqueeze(-1) * v_ref + steps.sum(dim=1)
    return nn.functional.normalize(q, dim=-1)


class Steerer(Protocol):
    """What training and evaluation actually require of a combiner.

    Typing the call sites against this rather than against the concrete model
    is what let the transformer variant be swapped out for cpas_mlp with
    nothing downstream to change.
    """

    def steer(
        self,
        v_ref: torch.Tensor,
        dirs: torch.Tensor,
        signs: torch.Tensor,
        mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        ...

    def __call__(
        self,
        v_ref: torch.Tensor,
        dirs: torch.Tensor,
        signs: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        ...


class FixedRule:
    """The gamma-weighted probe composition, wearing the Steerer interface.

    q = normalize(gamma * v_ref + sum(pos_dirs) - sum(neg_dirs)), i.e.
    `probes.compose_probe` with alpha = 1 and delta = 0. Having it satisfy the
    same protocol as the trained combiner is what lets the val benchmark and
    the exclusion-rerank sweep run over both without a second code path.
    """

    def __init__(self, gamma: float = 0.6) -> None:
        self.gamma = gamma

    def eval(self) -> "FixedRule":
        return self

    def train(self, mode: bool = True) -> "FixedRule":
        return self

    def steer(
        self,
        v_ref: torch.Tensor,
        dirs: torch.Tensor,
        signs: torch.Tensor,
        mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        gamma = torch.full((v_ref.shape[0],), self.gamma, device=v_ref.device)
        alpha = mask.to(dirs.dtype)
        return gamma, alpha, torch.zeros_like(dirs)

    def __call__(
        self,
        v_ref: torch.Tensor,
        dirs: torch.Tensor,
        signs: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        gamma, alpha, delta = self.steer(v_ref, dirs, signs, mask)
        return compose(v_ref, dirs, signs, gamma, alpha, delta)


def pad_queries(
    queries: list[tuple[list[int], list[int]]],
    directions: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Pack per-query attribute rows into padded (dirs, signs, mask) tensors.

    queries: list of (positive rows, negative rows) indexing `directions`;
    directions: (A, D) L2-normalized probe directions.
    """
    d = directions.shape[1]
    k = max((len(p) + len(n) for p, n in queries), default=0)
    dirs = torch.zeros(len(queries), k, d)
    signs = torch.zeros(len(queries), k)
    mask = torch.zeros(len(queries), k, dtype=torch.bool)
    for i, (pos, neg) in enumerate(queries):
        rows = pos + neg
        if not rows:
            continue
        dirs[i, : len(rows)] = directions[rows]
        signs[i, : len(pos)] = 1.0
        signs[i, len(pos) : len(rows)] = -1.0
        mask[i, : len(rows)] = True
    return dirs, signs, mask

In [ ]:
POS, NEG = 0, 1  # sign-embedding rows


class PerAttributeMLP(nn.Module):
    """Reference-conditioned steering of probe directions, MLP variant.

    d: embedding width; hidden: MLP width; sign_dim: width of the T+/T- tag;
    rank: rank of the delta factorization (the bend's degrees of freedom);
    delta_max: bound on the bend (0 disables bending); cross_attributes: when
    False the pooled context is zeroed, leaving conditioning on the reference
    only; gamma_init/alpha_init: the composition the untrained module
    reproduces.
    """

    def __init__(
        self,
        d: int = 512,
        hidden: int = 256,
        sign_dim: int = 64,
        rank: int = 32,
        delta_max: float = 0.3,
        gamma_init: float = 0.6,
        alpha_init: float = 1.0,
        cross_attributes: bool = True,
    ) -> None:
        super().__init__()
        self.delta_max = delta_max
        self.cross_attributes = cross_attributes
        self.sign_emb = nn.Embedding(2, sign_dim)
        nn.init.normal_(self.sign_emb.weight, std=0.02)
        self.trunk = nn.Sequential(
            nn.Linear(2 * d + sign_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )
        self.context = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
        )
        self.head_alpha = nn.Linear(hidden, 1)
        self.head_gamma = nn.Linear(hidden + d, 1)
        # delta = delta_max * tanh(U(V(h))): the bend lives in a rank-r subspace.
        self.delta_down = nn.Linear(hidden, rank)
        self.delta_up = nn.Linear(rank, d, bias=False)
        self._init_as_baseline(gamma_init, alpha_init)

    def _init_as_baseline(self, gamma_init: float, alpha_init: float) -> None:
        """Set the heads so the untrained module is the fixed-rule composition.

        delta_up is zeroed rather than delta_down, so delta = 0 at step 0 while
        delta_up still receives gradient and pulls delta_down into play on the
        first step - zeroing both would leave the bend permanently dead.
        """
        for head in (self.head_gamma, self.head_alpha):
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)
        nn.init.zeros_(self.delta_up.weight)
        # sigmoid(b) = gamma_init; softplus(b) = alpha_init
        self.head_gamma.bias.data.fill_(math.log(gamma_init / (1 - gamma_init)))
        self.head_alpha.bias.data.fill_(math.log(math.expm1(alpha_init)))

    def steer(
        self,
        v_ref: torch.Tensor,
        dirs: torch.Tensor,
        signs: torch.Tensor,
        mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Predict (gamma, alpha, delta) for a batch of queries.

        v_ref: (B, D) L2-normalized references.
        dirs:  (B, K, D) L2-normalized probe directions, padded with anything.
        signs: (B, K) +1 for T+ attributes, -1 for T-.
        mask:  (B, K) bool, True where the slot holds a real attribute.
        Returns gamma (B,), alpha (B, K), delta (B, K, D); alpha and delta are
        zeroed on padded slots.
        """
        k = dirs.shape[1]
        tags = self.sign_emb(torch.where(signs > 0, POS, NEG))
        tokens = torch.cat([v_ref.unsqueeze(1).expand(-1, k, -1), dirs, tags], dim=-1)
        h = self.trunk(tokens)  # (B, K, H)

        # Zero the padded rows before pooling: a padded slot must not leak into
        # any real attribute's context.
        keep = mask.unsqueeze(-1).to(h.dtype)
        h = h * keep
        total = h.sum(dim=1, keepdim=True)
        count = keep.sum(dim=1, keepdim=True)
        # Exclude self, so K=1 queries see an empty context.
        others = (total - h) / (count - keep).clamp(min=1.0)
        if not self.cross_attributes:
            others = torch.zeros_like(others)
        z = self.context(torch.cat([h, others], dim=-1))

        pooled = (total / count.clamp(min=1.0)).squeeze(1)  # (B, H)
        gamma = torch.sigmoid(self.head_gamma(torch.cat([pooled, v_ref], dim=-1)))
        alpha = nn.functional.softplus(self.head_alpha(z)).squeeze(-1)
        delta = self.delta_max * torch.tanh(self.delta_up(self.delta_down(z)))
        return gamma.squeeze(-1), alpha * keep.squeeze(-1), delta * keep

    def forward(
        self,
        v_ref: torch.Tensor,
        dirs: torch.Tensor,
        signs: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        """Composite query embeddings, (B, D) L2-normalized."""
        gamma, alpha, delta = self.steer(v_ref, dirs, signs, mask)
        return compose(v_ref, dirs, signs, gamma, alpha, delta)

## 5. Training Φ

### 5.1 Mining aligned to the criterion

There are no retrieval annotations for the train split, so training examples
are synthesised from CelebA labels: sample a reference, flip $k$ of its
attributes ($0\!\to\!1$ gives $T^+$, $1\!\to\!0$ gives $T^-$), then find real
images that are **correct answers under §3.1.1 itself** — the rule imported
from `criterion`, shared with the validation benchmark.

This sharing is the fix for a measured defect. An earlier miner elected its
target by agreement on ten hand-picked identity-proxy attributes with a
CLIP-similarity tiebreak — a *different* definition of "correct" than the one
the benchmark grades. It taught the model to change ~10 things where the
criterion tolerates 2, and its ideal exemplars were often not valid answers at
all. Three further consequences of using the real rule:

- **the target is drawn uniformly** from the valid set — §3.1.1 defines no
  ordering inside it, so electing a "best" target teaches a preference the
  criterion does not have;
- **negatives sit at the decision boundary**, one family per way of being
  wrong: *violators* (inside the Hamming ball, constraint broken) and
  *drifters* (constraints satisfied, drawn from the innermost shell outside the
  ball). The old miner picked the *most distant* qualifying image — the easiest
  negative of its family;
- **the rule is pure label logic**: no image features, which makes mining ~10×
  cheaper and lets the reference keep a dedicated negative slot (the image the
  model is most tempted to return).

Effect on the cosine-space row — identical architecture, budget and probes,
three seeds per miner, both recomputed inside a single run against the same
fixed-rule baseline (0.210):

| miner | R@10 | neg_R@10 | V@10 ↓ |
|---|---|---|---|
| identity-proxy target (old) | 0.267 ± 0.002 | 0.207 | 0.478 |
| **§3.1.1 target (delivered)** | **0.361 ± 0.013** | **0.319** | **0.256** |

**+0.094 R@10**, a 54% gain on the negation-bearing queries, and the violation
rate at 10 nearly halved. That last column is the sharpest evidence that the
old target definition was the defect and not a detail: at 0.478 the old miner
returned constraint-breaking images *more often than using no combiner at all*
(0.296 for the fixed rule). Training Φ against a notion of "correct" the
benchmark does not use did not merely fail to help — it actively taught the
model to violate the constraints it was conditioned on.

The new miner's seed spread is six times the old one's (σ 0.013 against 0.002).
Drawing the target uniformly from the valid set makes each run take a different
path through it, so in cosine space differences below ~0.03 are not resolved.

In [ ]:
N_NEGATIVES = 8   # per family; measured to be always available (S7 of the proposal)


@dataclass(frozen=True)
class MinedQuery:
    """One mined training example; all fields index the mining pool.

    Every field is required and positional on purpose: this replaced a
    three-field `Triplet`, and a missed call site should fail loudly rather than
    default to something plausible.
    """

    ref: int
    add: list[int]         # attribute rows to add    -> T+
    remove: list[int]      # attribute rows to remove -> T-
    target: int            # one valid answer, drawn uniformly
    violators: list[int]   # inside the ball, break a constraint
    drifters: list[int]    # satisfy the constraints, outside the ball


class Miner:
    """Samples attribute-flip queries from a labelled pool.

    labels: (N, A) bool attribute matrix for the mining pool. **Put it on the
        training device.** Every step of the rule is a pass over this matrix,
        once per sampled query, so leaving it on the CPU while the rest of the
        run is on the GPU costs roughly an order of magnitude in wall clock -
        measured the hard way.
    min_targets: reject a sampled flip set with fewer valid answers than this.
        The default mirrors the benchmark's own inclusion rule, so training
        queries are as hard as graded ones instead of systematically easier.
    n_negatives: how many to draw per family.
    weights: (A,) optional sampling weights over attributes. Rejection is
        attribute-dependent - `Mustache` survives it at 6.6%, `Black_Hair` at
        36.3% - so uniform sampling lets the filter choose the training
        distribution. Weights of 1/retention pre-compensate for it; None keeps
        uniform sampling and today's behaviour (proposal S7.1).
    """

    def __init__(
        self,
        labels: torch.Tensor,
        min_targets: int = MIN_TARGETS,
        n_negatives: int = N_NEGATIVES,
        max_hamming: int = MAX_HAMMING,
        weights: torch.Tensor | None = None,
        seed: int = 0,
    ) -> None:
        self.labels = labels.bool()
        self.min_targets = min_targets
        self.n_negatives = n_negatives
        self.max_hamming = max_hamming
        if weights is not None:
            if weights.shape != (labels.shape[1],):
                raise ValueError(
                    f"weights must have one entry per attribute: got "
                    f"{tuple(weights.shape)} for {labels.shape[1]} attributes"
                )
            weights = weights.double().clamp(min=0)
            if float(weights.sum()) <= 0:
                raise ValueError("weights must have a positive sum")
        self.weights = weights
        self.gen = torch.Generator().manual_seed(seed)

    # ------------------------------------------------------------------ steps

    def _sample_flips(self, ref: int, k: int) -> tuple[list[int], list[int]]:
        """Pick k attributes of the reference to flip, split by flip direction."""
        a = self.labels.shape[1]
        if self.weights is None:
            rows = torch.randperm(a, generator=self.gen)[:k]
        else:
            rows = torch.multinomial(self.weights, k, replacement=False,
                                     generator=self.gen)
        state = self.labels[ref]
        add = [int(r) for r in rows if not state[r]]
        remove = [int(r) for r in rows if state[r]]
        return add, remove

    def _draw(self, pool: torch.Tensor, count: int) -> list[int]:
        """`count` uniform draws without replacement from a boolean mask.

        Distinct positions are rejection-sampled rather than taken from a
        permutation of the candidate set. The drifter pool routinely holds tens
        of thousands of rows, and a randperm over those to keep eight of them,
        three times per example and 40k examples per epoch, dominated mining
        time. Duplicates matter here: a repeated negative would carry double
        weight in the softmax, which is the same reason padding is masked
        rather than repeated.
        """
        rows = pool.nonzero(as_tuple=True)[0]
        n = int(rows.numel())
        if n == 0:
            return []
        if count >= n:
            return rows.tolist()
        picked: set[int] = set()
        while len(picked) < count:
            for i in torch.randint(n, (2 * count,), generator=self.gen).tolist():
                picked.add(i)
                if len(picked) == count:
                    break
        idx = torch.tensor(sorted(picked), device=rows.device)
        return rows[idx].tolist()

    # ----------------------------------------------------------------- public

    def sample(self, k: int, max_tries: int = 50) -> MinedQuery | None:
        """Mine one query with k flipped attributes, or None if sampling failed.

        max_tries is deliberately generous. Acceptance is ~10.6% at k = 3, so a
        small budget would fail often, and `sample_batch` draws a fresh k on
        failure - which would quietly under-represent the hardest flip count.
        At 50 tries that leakage is under half a percent.
        """
        n = self.labels.shape[0]
        for _ in range(max_tries):
            ref = int(torch.randint(n, (1,), generator=self.gen))
            add, remove = self._sample_flips(ref, k)
            if not add and not remove:
                continue

            queried = add + remove
            s = satisfies(self.labels, add, remove)
            h = hamming_to(self.labels, self.labels[ref], queried)
            inside = h <= self.max_hamming

            valid = s & inside
            valid[ref] = False
            if int(valid.sum()) < self.min_targets:
                continue

            violators = (~s) & inside
            violators[ref] = False   # it owns a dedicated batch slot; not drawn twice
            if not bool(violators.any()):
                continue

            drifters = s & ~inside
            if not bool(drifters.any()):
                continue
            # The innermost non-empty shell: the candidates that miss the ball by
            # as little as possible are the ones that teach where its edge is.
            shell = int(h[drifters].min())
            drifters &= h == shell

            return MinedQuery(
                ref=ref,
                add=add,
                remove=remove,
                target=self._draw(valid, 1)[0],
                violators=self._draw(violators, self.n_negatives),
                drifters=self._draw(drifters, self.n_negatives),
            )
        return None

    def sample_batch(self, size: int, ks: tuple[int, ...] = (1,)) -> list[MinedQuery]:
        """Mine `size` queries, drawing each example's flip count from `ks`."""
        out: list[MinedQuery] = []
        while len(out) < size:
            k = int(ks[int(torch.randint(len(ks), (1,), generator=self.gen))])
            query = self.sample(k)
            if query is not None:
                out.append(query)
        return out


def retention_weights(retention: torch.Tensor, floor: float = 0.01) -> torch.Tensor:
    """Sampling weights that undo an attribute-dependent rejection rate.

    Rejection is a filter, so P_realised(a) is proportional to
    P_sampled(a) * retention(a). Sampling at 1/retention(a) therefore lands on a
    uniform realised distribution - inverse-propensity correction for a filter
    that cannot be removed.

    `retention` is the per-attribute column of scripts/measure_mining_rule.py,
    as a fraction. `floor` guards attributes that were never accepted in the
    measurement, which would otherwise divide by zero and swallow the whole
    sampling budget.

    This corrects the marginals only. `Male` and `Mustache` together are far
    worse than either alone, because correlated attributes drag each other out
    of the ball, and a per-attribute weight cannot see that.
    """
    return 1.0 / retention.double().clamp(min=floor)

### 5.2 The objective: InfoNCE with a false-negative mask

Each query must rank its mined target above every other in-batch target and
above its own mined negatives ($\tau = 0.05$):

$$ \mathcal{L} = -\log \frac{e^{\,q \cdot t^+ / \tau}}{\sum_{t \in \text{batch} \,\cup\, \text{mined}} e^{\,q \cdot t / \tau}} $$

With the criterion in hand, one training artefact becomes visible and fixable:
another row's target can be a *perfectly valid answer* to this row's query, and
vanilla InfoNCE would push the model **away from a correct answer**. Those
in-batch false negatives are detected with `valid_mask` and masked to
$-\infty$; padded negative slots are masked the same way rather than repeated,
since a duplicate would carry double weight in the softmax.

In [ ]:
@dataclass(frozen=True)
class Batch:
    """Model inputs and ranking candidates for one batch of mined queries."""

    v_ref: torch.Tensor      # (B, D)
    dirs: torch.Tensor       # (B, K, D)
    signs: torch.Tensor      # (B, K)
    mask: torch.Tensor       # (B, K) bool
    targets: torch.Tensor    # (B, D)
    negatives: torch.Tensor  # (B, 2M+1, D) violators | drifters | lazy
    neg_mask: torch.Tensor   # (B, 2M+1) bool, False on padded slots
    false_neg: torch.Tensor  # (B, B) bool, True where an in-batch target is valid here

    def to(self, device: str) -> "Batch":
        return Batch(*(t.to(device) for t in vars(self).values()))


def build_batch(
    queries: list[MinedQuery],
    features: torch.Tensor,
    directions: torch.Tensor,
    labels: torch.Tensor,
    max_hamming: int = MAX_HAMMING,
) -> Batch:
    """Gather features, padded attribute tensors and both masks.

    features / labels: (N, D) and (N, A) for the *mining pool*, aligned row for
    row - every index in a MinedQuery is a row of both.
    """
    dirs, signs, mask = pad_queries([(q.add, q.remove) for q in queries], directions)
    refs = [q.ref for q in queries]
    targets = [q.target for q in queries]

    b, d = len(queries), features.shape[1]
    n_viol = max((len(q.violators) for q in queries), default=0)
    n_drift = max((len(q.drifters) for q in queries), default=0)
    negatives = torch.zeros(b, n_viol + n_drift + 1, d,
                            dtype=features.dtype, device=features.device)
    neg_mask = torch.zeros(negatives.shape[:2], dtype=torch.bool,
                           device=features.device)
    for i, q in enumerate(queries):
        if q.violators:
            negatives[i, : len(q.violators)] = features[q.violators]
            neg_mask[i, : len(q.violators)] = True
        if q.drifters:
            end = n_viol + len(q.drifters)
            negatives[i, n_viol:end] = features[q.drifters]
            neg_mask[i, n_viol:end] = True
    # The reference owns the last slot unconditionally. It is formally a
    # violator - Hamming 0, every queried constraint broken - but it is also the
    # single highest-scoring image in the database whenever q collapses onto
    # v_ref, so its presence must not depend on a draw.
    negatives[:, -1] = features[refs]
    neg_mask[:, -1] = True

    # Validity is per row: row i has its own reference code and its own queried
    # columns, so there is no shared column set to vectorise over. B is ~1e3 and
    # each step is (B, A), so the loop is cheap and correct.
    target_labels, ref_labels = labels[targets], labels[refs]
    false_neg = torch.zeros(b, b, dtype=torch.bool, device=labels.device)
    for i, q in enumerate(queries):
        false_neg[i] = valid_mask(target_labels, ref_labels[i], q.add, q.remove,
                                  max_hamming)
    false_neg.fill_diagonal_(False)   # a row's own target is its positive

    return Batch(
        v_ref=features[refs],
        dirs=dirs,
        signs=signs,
        mask=mask,
        targets=features[targets],
        negatives=negatives,
        neg_mask=neg_mask,
        false_neg=false_neg,
    )


def _logits(
    q: torch.Tensor,
    targets: torch.Tensor,
    negatives: torch.Tensor,
    neg_mask: torch.Tensor | None,
    false_neg: torch.Tensor | None,
) -> torch.Tensor:
    """(B, B + M) candidate scores, masked slots sent to -inf."""
    in_batch = q @ targets.T
    if false_neg is not None:
        in_batch = in_batch.masked_fill(false_neg, float("-inf"))
    mined = torch.einsum("bd,bmd->bm", q, negatives)
    if neg_mask is not None:
        mined = mined.masked_fill(~neg_mask, float("-inf"))
    return torch.cat([in_batch, mined], dim=1)


def infonce_loss(
    q: torch.Tensor,
    targets: torch.Tensor,
    negatives: torch.Tensor,
    neg_mask: torch.Tensor | None = None,
    false_neg: torch.Tensor | None = None,
    tau: float = 0.05,
) -> torch.Tensor:
    """InfoNCE over in-batch targets plus each query's own mined negatives.

    q, targets: (B, D) L2-normalized; negatives: (B, M, D). Row i's positive is
    targets[i]. With both masks left at None this is the plain unmasked loss.
    """
    logits = _logits(q, targets, negatives, neg_mask, false_neg) / tau
    labels = torch.arange(q.shape[0], device=q.device)
    return nn.functional.cross_entropy(logits, labels)


def recall_at_1(q: torch.Tensor, batch: Batch) -> float:
    """Fraction of queries ranking their target above every batch candidate."""
    scores = _logits(q, batch.targets, batch.negatives, batch.neg_mask,
                     batch.false_neg)
    hit = scores.argmax(dim=1) == torch.arange(q.shape[0], device=q.device)
    return float(hit.float().mean())


def run_epoch(
    model: Steerer,
    queries: list[MinedQuery],
    features: torch.Tensor,
    directions: torch.Tensor,
    labels: torch.Tensor,
    optimizer: torch.optim.Optimizer | None = None,
    batch_size: int = 256,
    tau: float = 0.05,
    device: str = "cpu",
) -> tuple[float, float]:
    """One pass over `queries`; trains when an optimizer is given.

    Returns (mean loss, mean batch recall@1).
    """
    train = optimizer is not None
    model.train(train)
    losses, recalls = [], []
    for start in range(0, len(queries) - 1, batch_size):
        chunk = queries[start : start + batch_size]
        if len(chunk) < 2:  # InfoNCE needs at least one in-batch negative
            continue
        batch = build_batch(chunk, features, directions, labels).to(device)
        with torch.set_grad_enabled(train):
            q = model(batch.v_ref, batch.dirs, batch.signs, batch.mask)
            loss = infonce_loss(q, batch.targets, batch.negatives,
                                batch.neg_mask, batch.false_neg, tau=tau)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        losses.append(float(loss.detach()))
        recalls.append(recall_at_1(q.detach(), batch))
    n = max(len(losses), 1)
    return sum(losses) / n, sum(recalls) / n

### 5.3 Training run

Selection is val-benchmark R@10, not the mining proxy — the proxy tracks the
true metric poorly. The validation benchmark replicates the §3.1.1 rule over a
held-out 10% slice of the train pool (details in §6); its builder and scorer
are defined here because training selects on them.

In [ ]:
def build_val_benchmark(
    labels: torch.Tensor,
    query_specs: list[tuple[list[int], list[int]]],
    directions: torch.Tensor,
    per_query: int = 200,
    min_gt: int = 3,
    seed: int = 0,
    max_hamming: int = MAX_HAMMING,
) -> list[dict]:
    """Held-out replica of the eval benchmark for checkpoint selection.

    Reproduces the assignment's ground-truth rule exactly (S3.1.1: constraints
    satisfied, plus Hamming distance <= 2 to the reference over the non-queried
    attributes) but over a held-out image pool, so it tracks the true R@10
    without ever touching the test references or the test ground truth. Only the
    query *shapes* are shared with the benchmark, which is the distribution we
    are graded on.

    An earlier version required exact agreement on ten identity-proxy
    attributes. That is a different task: it selected checkpoints and tuned
    hyperparameters against ground truth the benchmark does not use, which is
    why validation gains did not transfer to test.

    The rule itself is `src/criterion.valid_mask`, shared with the miner: the
    two used to carry separate implementations, and only one of them was right.

    labels: (N, A) bool for the val pool (references and database are this pool);
    query_specs: (positive rows, negative rows) per query, indexing `labels`;
    directions: (A, D) probe dirs.
    Returns one task dict per query with precomputed reference indices, a
    (R, N) ground-truth mask, and the padded (dirs, signs, mask) model inputs.
    """
    gen = torch.Generator().manual_seed(seed)
    n = labels.shape[0]
    tasks = []
    for pos_rows, neg_rows in query_specs:
        refs, masks = [], []
        for r in torch.randperm(n, generator=gen).tolist():
            gt = valid_mask(labels, labels[r], pos_rows, neg_rows, max_hamming)
            gt[r] = False
            if int(gt.sum()) >= min_gt:
                refs.append(r)
                masks.append(gt)
            if len(refs) >= per_query:
                break
        if not refs:
            continue
        refs_t = torch.tensor(refs)
        dirs, signs, mask = pad_queries([(pos_rows, neg_rows)] * len(refs), directions)
        tasks.append({
            "refs": refs_t,
            "gt_mask": torch.stack(masks),
            "dirs": dirs, "signs": signs, "mask": mask,
            "pos_rows": list(pos_rows), "neg_rows": list(neg_rows),
        })
    return tasks


@torch.no_grad()
def score_val_benchmark(
    model: Steerer,
    db: torch.Tensor,
    tasks: list[dict],
    k: int = 10,
    rerank: Rerank | None = None,
) -> float:
    """Mean Recall@k of `model` over a val benchmark from `build_val_benchmark`.

    db: (N, D) L2-normalized features of the val pool (the ranking database);
    a query counts as a hit when any ground-truth image is in its top-k.

    `rerank` applies the exclusion penalty to the val ranking; its `db_probs`
    must be the val pool's, not the test split's. This is the surface the
    lambda sweep tunes on - the 14 test queries are never touched.
    """
    model.eval()
    device = db.device
    hits, total = 0, 0
    for task in tasks:
        refs = task["refs"].to(device)
        q = model(
            db[refs], task["dirs"].to(device),
            task["signs"].to(device), task["mask"].to(device),
        )
        if rerank is None:
            sims = q @ db.T
        else:
            sims = rerank.scores(q, db, task["pos_rows"], task["neg_rows"])
        sims[torch.arange(refs.shape[0], device=device), refs] = float("-inf")
        top = sims.topk(k, dim=1).indices
        hit = task["gt_mask"].to(device).gather(1, top).any(dim=1)
        hits += int(hit.sum())
        total += refs.shape[0]
    return hits / max(total, 1)

Warmup on single-attribute flips, then $k \in \{1,2,3\}$, fresh triplets
re-mined every epoch, early stopping on val R@10. The cell trains only when the
released checkpoint is absent, and logs a history CSV so the learning curves
below always have a source.

In [ ]:
CPAS_HISTORY = Path("results/cpas_training_history.csv")

if CFG["cpas_path"].is_file():
    ckpt = torch.load(CFG["cpas_path"], map_location="cpu", weights_only=True)
    cfg = dict(ckpt.get("config", {})); cfg.pop("arch", None)
    combiner = PerAttributeMLP(**cfg)
    combiner.load_state_dict(ckpt["state_dict"])
    combiner.eval()
    print(f"loaded {CFG['cpas_path'].name}: epoch {ckpt.get('epoch')}, "
          f"seed {ckpt.get('seed')}, config {cfg}")
else:
    # ~2h15 on a T4. These are the exact settings that produced the released
    # checkpoint, not the script defaults: the epoch cap was raised to 150
    # because the model had not converged at 50, patience with it, and triplets
    # are re-mined every third epoch rather than every epoch - mining is over
    # 90% of an epoch's cost, and the report's ablation found per-epoch
    # re-mining bought nothing for it.
    WARMUP, EPOCHS, N_TRIPLETS, BATCH, LR, PATIENCE = 5, 145, 40_000, 1024, 1e-4, 20
    REMINE_EVERY = 3
    torch.manual_seed(0)
    perm = torch.randperm(pool_features.shape[0],
                          generator=torch.Generator().manual_seed(0))
    split = int(pool_features.shape[0] * 0.9)
    train_pool = pool_features[perm[:split]].to(DEVICE)
    val_pool = pool_features[perm[split:]].to(DEVICE)
    train_labels_d = pool_labels[perm[:split]].to(DEVICE)
    miner = Miner(train_labels_d, seed=0)
    specs = [([attr_index[a] for a in _parse(e["query"])[0]],
              [attr_index[a] for a in _parse(e["query"])[1]]) for e in annotations]
    val_tasks = build_val_benchmark(pool_labels[perm[split:]], specs, directions)
    combiner = PerAttributeMLP().to(DEVICE)
    opt = torch.optim.Adam(combiner.parameters(), lr=LR)
    dirs_d = directions.to(DEVICE)
    best, history = {"r10": -1.0, "state": None, "epoch": -1}, []
    triplets = None
    with torch.enable_grad():
        for epoch in range(WARMUP + EPOCHS):
            phase, ks = ("warmup", (1,)) if epoch < WARMUP else ("main", (1, 2, 3))
            # Reproduces scripts/train_cpas.py exactly, quirk included: the
            # first main epoch is not a re-mining epoch, so it still trains on
            # the k = 1 set mined during warmup.
            if triplets is None or epoch % REMINE_EVERY == 0:
                triplets = miner.sample_batch(N_TRIPLETS, ks=ks)
            loss, proxy = run_epoch(combiner, triplets, train_pool, dirs_d,
                                    train_labels_d, optimizer=opt,
                                    batch_size=BATCH, device=DEVICE)
            r10 = score_val_benchmark(combiner, val_pool, val_tasks)
            history.append({"epoch": epoch, "phase": phase, "loss": loss,
                            "proxy_r1": proxy, "val_r10": r10})
            print(f"epoch {epoch:3d} [{phase}]  loss {loss:.4f}  "
                  f"proxy r@1 {proxy:.3f}  val R@10 {r10:.4f}", flush=True)
            if r10 > best["r10"]:
                best = {"r10": r10, "epoch": epoch,
                        "state": {k: t.detach().cpu().clone()
                                  for k, t in combiner.state_dict().items()}}
            elif phase == "main" and epoch - max(best["epoch"], WARMUP - 1) >= PATIENCE:
                print("early stop"); break
    combiner.load_state_dict(best["state"]); combiner.eval(); combiner.cpu()
    torch.save({"state_dict": best["state"], "attributes": attributes,
                "val_r10": best["r10"], "epoch": best["epoch"],
                "config": {"delta_max": 0.3, "rank": 32, "cross_attributes": True},
                "seed": 0}, CFG["cpas_path"])
    pd.DataFrame(history).to_csv(CPAS_HISTORY, index=False)
    print(f"best epoch {best['epoch']} (val R@10 {best['r10']:.4f})")

### Learning curves

Loss, the mining proxy (batch recall@1), and the selection signal (val R@10).
The two do not move together — the proxy saturates early while val R@10 keeps
improving — which is precisely why checkpoint selection uses the benchmark
replica and not the proxy.

In [ ]:
if CPAS_HISTORY.is_file():
    hist = pd.read_csv(CPAS_HISTORY)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, col, label in zip(axes, ["loss", "proxy_r1", "val_r10"],
                              ["InfoNCE loss", "batch recall@1 (proxy)",
                               "val R@10 (selection signal)"]):
        ax.plot(hist["epoch"], hist[col], lw=1.5)
        if "phase" in hist.columns and (hist["phase"] == "warmup").any():
            ax.axvline(hist[hist["phase"] == "main"]["epoch"].min() - 0.5,
                       ls="--", lw=0.8, color="gray")
        ax.set_xlabel("epoch"); ax.set_title(label, fontsize=10)
    best_row = hist.loc[hist["val_r10"].idxmax()]
    axes[2].scatter([best_row["epoch"]], [best_row["val_r10"]], color="tab:red",
                    zorder=3, label=f"selected: epoch {int(best_row['epoch'])}")
    axes[2].legend(fontsize=8)
    fig.tight_layout(); plt.show()
else:
    print(f"{CPAS_HISTORY} not found - produced by the training cell above; "
          "the released checkpoint was trained by scripts/train_cpas.py with "
          "the same loop.")

## 6. Experimental setup

| | |
|---|---|
| retrieval database | official CelebA **test** split, 19,962 images, embedded once and frozen |
| training pool | **train** split (162,770); CPAS-MLP holds out a 10% slice as its validation pool |
| selection surface | **valid** split for the attribute head epoch + thresholds; held-out train slice for $\lambda, w$ and CPAS checkpoints |
| benchmark | the 14 mandatory queries of `celeba_evaluation.json`, R@K / P@K at $K \in \{1,5,10\}$, averaged over each query's source images |

**The validation benchmark replicates §3.1.1 exactly** — same rule, same query
shapes, over held-out images — via `build_val_benchmark` (already defined with
the training code, imported from the shared `criterion`). An earlier version
used a ten-attribute identity-proxy rule; that is a different task, and gains
selected on it did not transfer. The test queries and test ground truth are
never touched by any tuning step.

**Extra metrics** beyond the required R@K/P@K: **V@10**, the fraction of
returned top-10 images breaking a constraint (computed from true labels — the
quantity the constraint term targets, which R@K cannot stand in for), and
**neg-R@10**, mean R@10 over the queries carrying a negation, where cosine
methods historically fail.

**Reading the numbers.** Nothing in the delivered scoring path is trained at
benchmark time, so given fixed features every row below is deterministic. The
resolution rule applies to *trained* comparisons, where the delivered miner's
seed spread is σ = 0.013 over three seeds (§5.1) — so cosine-space deltas
below ~0.03 are not resolved. Deterministic scoring rows differ meaningfully
at much smaller deltas.
All baseline rows are recomputed live in this notebook, so every comparison is
within-run.

In [ ]:
KS = (1, 5, 10)


VIOLATION_K = 10




def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int,
) -> dict:
    """Recall@K (binary hit rate) and Precision@K for a single source image.

    Same semantics as the function provided in the course skeleton.
    """
    top_k = retrieved_indices[:k]
    num_hits = len(set(top_k) & set(ground_truth_indices))
    return {
        f"Recall@{k}": 1 if num_hits > 0 else 0,
        f"Precision@{k}": num_hits / k,
    }


def violation_rate(
    order: torch.Tensor,
    labels: torch.Tensor,
    pos_rows: list[int],
    neg_rows: list[int],
    k: int = VIOLATION_K,
) -> float:
    """Fraction of returned top-k images that break at least one constraint.

    This is what the exclusion penalty directly targets, and R@k cannot stand in
    for it: R@k can improve for reasons unrelated to exclusion, so a rise in
    R@k without a fall here leaves the mechanism claim unsupported
    (docs/method.md S8).

    order: (S, N) retrieved indices, best first; labels: (N, A) bool.
    """
    top = order[:, :k]
    got = labels.to(order.device)[top]  # (S, k, A)
    bad = torch.zeros(top.shape, dtype=torch.bool, device=order.device)
    if pos_rows:
        bad |= ~got[:, :, pos_rows].all(dim=-1)
    if neg_rows:
        bad |= got[:, :, neg_rows].any(dim=-1)
    return float(bad.float().mean())


def negation_subset(df: pd.DataFrame, column: str = "R@10") -> float:
    """Mean of `column` over the benchmark queries carrying a negation.

    The overall mean is diluted by the positive-only queries that the
    negation-aware changes are not meant to help - 6 of the 14 - so the subset
    is reported separately (docs/method.md S8).
    """
    rows = df[(df["query"] != "MEAN") & df["query"].str.contains("-")]
    return float(rows[column].mean()) if len(rows) else float("nan")


def _query_row(
    entry: dict,
    order: torch.Tensor,
    source_indices: list[int],
    labels: torch.Tensor | None = None,
    pos_rows: list[int] | None = None,
    neg_rows: list[int] | None = None,
) -> dict:
    """Per-query metrics row: R@K / P@K averaged over the source images.

    With `labels` given, the row also carries the top-10 violation rate; the
    column is absent otherwise, so callers that never pass labels produce the
    same table they always did.
    """
    sums = {f"R@{k}": 0.0 for k in KS} | {f"P@{k}": 0.0 for k in KS}
    for row_idx, src in enumerate(source_indices):
        retrieved = order[row_idx, : max(KS)].tolist()
        targets = entry["ground_truth"][str(src)]
        for k in KS:
            m = evaluate_retrieval(retrieved, targets, k)
            sums[f"R@{k}"] += m[f"Recall@{k}"]
            sums[f"P@{k}"] += m[f"Precision@{k}"]
    n = len(source_indices)
    row = {"query": entry["query"], "sources": n} | {
        key: val / n for key, val in sums.items()
    }
    if labels is not None:
        row[f"V@{VIOLATION_K}"] = violation_rate(
            order, labels, pos_rows or [], neg_rows or [], k=VIOLATION_K
        )
    return row


def _with_mean_row(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    metric_cols = [f"R@{k}" for k in KS] + [f"P@{k}" for k in KS]
    if f"V@{VIOLATION_K}" in df.columns:
        metric_cols.append(f"V@{VIOLATION_K}")
    mean_row = {"query": "MEAN", "sources": df["sources"].sum()}
    mean_row |= df[metric_cols].mean().to_dict()
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)
    return df[["query", "sources", *metric_cols]]

The evaluation harness: every method below is a `score_fn(entry, refs, pos, neg)
→ (R, N)` scores matrix; ranking, reference exclusion and metrics are shared.
`evaluate_retrieval` is semantically the function provided in the course
skeleton.

In [ ]:
queries = [parse_query(e["query"]) for e in annotations]
query_rows = [([attr_index[a] for a in p], [attr_index[a] for a in n])
              for p, n in queries]
sources_per_query = [[int(k) for k in e["ground_truth"]] for e in annotations]


def top_k_order(scores, srcs, k=max(KS)):
    """(R, k) database indices, best first, the reference itself excluded."""
    scores = scores.clone()
    scores[torch.arange(scores.shape[0]), torch.tensor(srcs)] = float("-inf")
    return scores.topk(k, dim=1).indices


def benchmark(score_fn, name):
    """Per-query metric table for one scoring rule, mean row appended."""
    rows = []
    for entry, (pr, nr), srcs in zip(annotations, query_rows, sources_per_query):
        order = torch.cat([
            top_k_order(score_fn(entry, srcs[i:i + SOURCE_CHUNK], pr, nr),
                        srcs[i:i + SOURCE_CHUNK])
            for i in range(0, len(srcs), SOURCE_CHUNK)])
        rows.append(_query_row(entry, order, srcs, labels, pr, nr))
    df = _with_mean_row(rows).assign(method=name)
    mean = df[df["query"] == "MEAN"].iloc[0]
    print(f"{name:52s} R@10 {mean['R@10']:.4f}  V@10 {mean['V@10']:.4f}")
    return df

Predictors and the delivered query pipeline. `reference_code` implements the
soft reference of §3; `cosine_term` uses the fusion module's composed query —
with the checkpoint absent it would fall back to the raw reference, and §7
ablates both.

In [ ]:
# Predicted probabilities and thresholded codes for the whole database, once.
probe_probs = torch.sigmoid(features @ W.T + B)
head_probs = torch.sigmoid(head(features))
predictors = {
    "linear probe": (probe_probs, probe_probs > 0.5),
    "MLP head": (head_probs, head_probs > head_thresholds),
}


def reference_code(probs, code_m, refs, pr, nr, soft=None):
    """Target code for a block of references; soft keeps the probabilities."""
    soft = CFG["soft_reference"] if soft is None else soft
    return target_code((probs if soft else code_m)[refs], pr, nr)


def cosine_term(refs, pr, nr, source="cpas"):
    """(N, R) similarity of the composite query to the database."""
    if source == "vref":
        return features @ features[refs].T
    dirs, signs, mask = pad_queries([(pr, nr)] * len(refs), directions)
    return features @ combiner(features[refs], dirs, signs, mask).T


def attribute_score_fn(predictor, soft=None, w_cos=None, cosine_source="cpas",
                       lam=None):
    probs, code_m = predictors[predictor]
    w = CFG["w_cos"] if w_cos is None else w_cos
    lam = CFG["lam_constraint"] if lam is None else lam
    def fn(entry, refs, pr, nr):
        cos = cosine_term(refs, pr, nr, cosine_source) if w else None
        return attribute_scores(probs, code_m,
                                reference_code(probs, code_m, refs, pr, nr, soft),
                                pr, nr, lam_constraint=lam, cosine=cos, w_cos=w)
    return fn

Optional: re-run the $(\lambda, w)$ sweep on the held-out validation slice
instead of trusting the hardcoded values. Off by default (`RUN_SWEEP`); the
selected pair is stated in §0 with its provenance.

In [ ]:
if RUN_SWEEP:
    perm = torch.randperm(pool_features.shape[0],
                          generator=torch.Generator().manual_seed(0))
    val_rows_ = perm[int(pool_features.shape[0] * 0.9):]
    vf, vl = pool_features[val_rows_], pool_labels[val_rows_]
    v_probs = torch.sigmoid(head(vf))
    v_code = v_probs > head_thresholds
    val_refs = torch.randperm(vf.shape[0],
                              generator=torch.Generator().manual_seed(0))[:200]
    def val_r10(lam, w):
        hits = total = 0
        for (pr, nr) in query_rows:
            others = [a for a in range(N_ATTR) if a not in set(pr) | set(nr)]
            sat = satisfies(vl, pr, nr)
            rest = vl[:, others]
            dirs, signs, mask = pad_queries([(pr, nr)] * len(val_refs), directions)
            cos = vf @ combiner(vf[val_refs], dirs, signs, mask).T if w else None
            order = rank_by_attributes(
                v_probs, v_code, target_code(v_probs[val_refs], pr, nr), pr, nr,
                exclude=val_refs.tolist(), lam_constraint=lam, cosine=cos, w_cos=w)
            for row, r in enumerate(val_refs.tolist()):
                gt = sat & ((rest != rest[r]).sum(dim=1) <= MAX_HAMMING)
                gt[r] = False
                if int(gt.sum()) < 3: continue
                hits += bool(gt[order[row, :10]].any()); total += 1
        return hits / max(total, 1)
    for lam in (1.0, 4.0, 16.0, 100.0):
        for w in (0.0, 1.0, 3.0, 10.0):
            print(f"lam {lam:<6} w {w:<5} val R@10 {val_r10(lam, w):.4f}")

## 7. Results

### 7.1 The ladder: from the required baseline to the delivered method

Four rows, one change at a time. All recomputed live against the full test
split.

In [ ]:
prompt_cache: dict[str, torch.Tensor] = {}
def text_dirs(names):
    missing = [a for a in names if a not in prompt_cache]
    if missing:
        embs = encoder.encode_texts([PROMPTS[a] for a in missing])
        prompt_cache.update(dict(zip(missing, embs)))
    return torch.stack([prompt_cache[a] for a in names]) if names \
        else torch.zeros(0, features.shape[1])


def prompt_score_fn(entry, refs, pr, nr):
    pos, neg = parse_query(entry["query"])
    edit = text_dirs(pos).sum(0) - text_dirs(neg).sum(0)
    q = features[refs] + edit                       # gamma = 1: the vanilla baseline
    return (q / q.norm(dim=1, keepdim=True)) @ features.T


def probe_comp_score_fn(entry, refs, pr, nr):
    edit = directions[pr].sum(0) - directions[nr].sum(0)
    q = 0.6 * features[refs] + edit                 # tuned gamma = 0.6
    return (q / q.norm(dim=1, keepdim=True)) @ features.T


def cpas_cosine_score_fn(entry, refs, pr, nr):
    return cosine_term(refs, pr, nr, "cpas").T


ladder = [
    benchmark(prompt_score_fn, "1. zero-shot prompt arithmetic (gamma=1)"),
    benchmark(probe_comp_score_fn, "2. probe-direction composition (gamma=0.6)"),
    benchmark(cpas_cosine_score_fn, "3. CPAS-MLP fusion, cosine ranking"),
    benchmark(attribute_score_fn("MLP head"),
              "4. DELIVERED: attribute-space score + CPAS-MLP"),
]

In [ ]:
metric_cols = [f"R@{k}" for k in KS] + [f"P@{k}" for k in KS] + ["V@10"]
summary = pd.concat([d[d["query"] == "MEAN"] for d in ladder]) \
            .set_index("method")[metric_cols]
summary["neg_R@10"] = [negation_subset(d) for d in ladder]
summary.style.format("{:.3f}").set_caption(
    "MEAN over the 14 benchmark queries, full test-split database")

Reading the ladder: swapping text prompts for in-space probe directions roughly
doubles R@10; the trained fusion module adds ~65% over the fixed rule *within*
cosine ranking; and moving the ranking into attribute space — same predictors,
no additional training — adds more than everything before it combined. The
compensatory→conjunctive shift shows in V@10, which only the constraint-aware
rows reduce.

### 7.2 Ablation of the delivered score

Two knobs of the score, one at a time, on both predictors. `soft` is the
reference representation of §3; the cosine column is what $q$ the third term
uses. Every row shares the same $\lambda = 4$; $w = 1$ wherever a cosine term
is present.

In [ ]:
grid = []
for pred in ("MLP head", "linear probe"):
    for soft in (False, True):
        for cos_src, w in (("none", 0.0), ("vref", 1.0), ("cpas", 1.0)):
            name = (f"{pred} | {'soft' if soft else 'hard'} reference | "
                    f"cosine: {cos_src}")
            df = benchmark(attribute_score_fn(pred, soft=soft, w_cos=w,
                                              cosine_source=cos_src), name)
            mean = df[df["query"] == "MEAN"].iloc[0]
            grid.append({"predictor": pred, "reference": "soft" if soft else "hard",
                         "cosine": cos_src, "R@10": mean["R@10"],
                         "P@10": mean["P@10"], "V@10": mean["V@10"],
                         "neg_R@10": negation_subset(df)})
grid = pd.DataFrame(grid)
grid.style.format({c: "{:.4f}" for c in ("R@10", "P@10", "V@10", "neg_R@10")}) \
    .set_caption("Score ablation: reference representation x cosine source")

What the grid establishes (numbers from the executed run above):

- **The soft reference is the single largest knob** after the space change
  itself: ~+0.04 R@10 on both predictors, for zero parameters and zero
  training. It is a strict generalisation, so the gain is attributable to
  keeping the predictor's uncertainty in the ranking.
- **The MLP head's +0.003 bit accuracy shows up as R@10**, consistent with the
  ~7× amplification measured earlier — the leading-indicator relationship
  holds.
- **The cosine term's contribution is small and shrinks as the attribute terms
  improve** — the pattern §8 explains and then tests directly.

### 7.3 Per-query behaviour

In [ ]:
delivered = ladder[3]
per_q = delivered[delivered["query"] != "MEAN"].drop_duplicates("query") \
    .sort_values("R@10")
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["tab:red" if "-" in q else "tab:blue" for q in per_q["query"]]
ax.barh(per_q["query"], per_q["R@10"], color=colors)
ax.axvline(float(delivered[delivered["query"] == "MEAN"]["R@10"].iloc[0]),
           ls="--", lw=1, color="k")
ax.set_xlabel("R@10 (delivered configuration)")
ax.set_title("Per-query R@10 — red: queries carrying a negation", fontsize=10)
fig.tight_layout(); plt.show()
per_q[["query", "sources", "R@1", "R@5", "R@10", "P@10", "V@10"]] \
    .style.format({c: "{:.3f}" for c in ("R@1", "R@5", "R@10", "P@10", "V@10")})

Two structural notes on the benchmark itself, visible in the table: `-Young`
appears **twice** among the 14 mandatory queries, so the MEAN row weights it
double (we follow the JSON as instructed and note it); and the hardest queries
combine a negation with correlated attributes
(`+Wearing_Lipstick, -Heavy_Makeup, +Smiling` has 34 sources and the smallest
ground-truth sets — flipping makeup off while keeping lipstick on is rare in
the database itself).

### 7.4 Qualitative examples

In [ ]:
def show_retrieval(query_idx: int, source_key: int, k: int = 5):
    """Reference + top-k of the delivered method; green = valid ground truth."""
    entry = annotations[query_idx]
    pr, nr = query_rows[query_idx]
    scores = attribute_score_fn("MLP head")(entry, [source_key], pr, nr)
    order = top_k_order(scores, [source_key], k=k)[0].tolist()
    gt = set(entry["ground_truth"][str(source_key)])
    fig, axes = plt.subplots(1, k + 1, figsize=(2.2 * (k + 1), 2.6))
    axes[0].imshow(celeba[source_key][0]); axes[0].set_title("reference", fontsize=9)
    for ax, idx in zip(axes[1:], order):
        ax.imshow(celeba[idx][0])
        hit = idx in gt
        ax.set_title("valid" if hit else "invalid", fontsize=9,
                     color="green" if hit else "red")
        for s in ax.spines.values():
            s.set_edgecolor("green" if hit else "red"); s.set_linewidth(3)
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'"{entry["query"]}"', fontsize=11)
    fig.tight_layout(); plt.show()


def first_source(query_idx: int, want_hit: bool):
    """A source whose top-10 is (or is not) a hit, for honest examples."""
    entry = annotations[query_idx]
    pr, nr = query_rows[query_idx]
    for src in sources_per_query[query_idx][:200]:
        scores = attribute_score_fn("MLP head")(entry, [src], pr, nr)
        top = top_k_order(scores, [src], k=10)[0].tolist()
        if bool(set(top) & set(entry["ground_truth"][str(src)])) == want_hit:
            return src
    return sources_per_query[query_idx][0]

Q_SUCCESS = 1    # +Eyeglasses: the strongest single-attribute query
Q_FAILURE = 13   # +Wearing_Lipstick, -Heavy_Makeup, +Smiling: the hardest query
show_retrieval(Q_SUCCESS, first_source(Q_SUCCESS, want_hit=True))
show_retrieval(Q_FAILURE, first_source(Q_FAILURE, want_hit=False))

The failure case is instructive rather than random: lipstick-without-makeup
sits on the boundary the predictors are worst at (`Heavy_Makeup` and
`Wearing_Lipstick` are strongly correlated in the data), so the reference code
itself is often wrong on exactly the queried bits — an error class the score
inherits from the predictor and cannot repair.

## 8. What the cosine term can and cannot contribute

Our starting hypothesis for keeping $w\,(q \cdot d)$ in the score was that raw
embedding similarity preserves information about the source that the 40
attributes do not encode — background, pose, lighting, identity beyond
attributes — and that dropping it would lose those images.

**The metric cannot reward that information.** Under §3.1.3 the ground truth is
a *set* $G$ and both metrics are set operations:
$\text{R@K} = \mathbb{1}[|R_K \cap G| > 0]$,
$\text{P@K} = |R_K \cap G| / K$. Swapping the ranks of two images that are both
in $G$ changes neither — **no correct answer is more correct than another**.
Since all of §3.1.1 is attribute-defined, information *orthogonal* to the
attribute code can only reorder candidates whose relative order is not graded.

It does not follow that such information is free. The cosine term does not
reorder within $G$ alone: it perturbs the score of *every* candidate. Adding a
component uncorrelated with the criterion to an otherwise correctly sorted
ranking pushes ground-truth images out of the top $K$ at random, so orthogonal
information does not contribute zero — **it costs**. §7.2 measures exactly
that: with the MLP head, the raw-reference cosine ($q = v_{ref}$, R@10 0.525)
scores *below* dropping the term entirely (0.540).

The only channel through which a cosine term can *help* is therefore its
correlation with the attribute code — a **second, coarser estimate of the code
itself**, redundant with the first term and useful only where the predictor is
weak. It is also why the composed query beats the raw reference (0.542): $q$
has been moved toward the requested attributes, so it estimates the *target*
code, while $v_{ref}$ still estimates the reference's — which on a queried
attribute is precisely the wrong one.

That yields a testable prediction: **the optimal $w$ should grow as the
predictor degrades, and vanish as it improves.** The grid in §7.2 already
points that way (the cosine column matters least in the best rows); the
experiment below tests it directly by corrupting the predictor at increasing
rates and re-selecting $w$ on validation each time.

In [ ]:
DEGRADED = sorted(Path("results").glob("degraded_predictor_sweep_s*.csv"))
# One file per CPAS-MLP seed, produced by scripts/run_degraded_sweep.py. Long
# format: a row per (predictor, corruption, corruption seed, q_source, w_cos).
# The full grid is kept rather than the argmax because with a validation SE
# around 0.01 the argmax of a flat curve is a noise generator.
if not DEGRADED:
    raise SystemExit(
        "missing results/degraded_predictor_sweep_s*.csv - produce them with\n"
        "  for s in 0 1 2; do python scripts/run_degraded_sweep.py \\\n"
        "      --w-grid 0 1 3 10 30 100 300 \\\n"
        "      --checkpoint results/mlp_final_s$s.pt \\\n"
        "      --out results/degraded_predictor_sweep_s$s.csv; done")

deg = pd.concat([pd.read_csv(f).assign(cpas_seed=int(f.stem.rsplit("_s", 1)[1]))
                 for f in DEGRADED], ignore_index=True)
deg["corruption"] = deg["corruption"].fillna(-1.0)   # -1 marks the real anchor
# The w = 0 and q = v_ref rows never touch the combiner, so they must be
# identical across the three files - a free check that the validation slice is
# rebuilt deterministically. Keeping all three copies would triple their weight
# and report a spurious sd of exactly zero, so only the first is retained.
shared = deg[deg.q_source != "cpas"]
per_seed = [g.drop(columns="cpas_seed").reset_index(drop=True)
            for _, g in shared.groupby("cpas_seed")]
assert all(g.equals(per_seed[0]) for g in per_seed), \
    "the checkpoint-independent rows differ between files"
deg = deg[(deg.q_source == "cpas") | (deg.cpas_seed == deg.cpas_seed.min())]
KEY = ["predictor", "corruption", "seed", "cpas_seed"]

# w = 0 is the reference point of every condition: the score with no embedding
# term at all. A "best w" scoring below it is not a gain, and reading the argmax
# alone would hide that - it is exactly what happens to the anchor.
w0 = deg[deg.q_source == "none"].set_index(KEY[:3])["val_R10"].rename("R10_w0")
best = (deg[deg.q_source != "none"]
        .loc[lambda d: d.groupby(KEY + ["q_source"])["val_R10"].idxmax()]
        .set_index(KEY + ["q_source"]))
tab = best.join(w0, on=KEY[:3])
tab["gain"] = tab["val_R10"] - tab["R10_w0"]

summary = (tab.reset_index()
           .groupby(["predictor", "corruption", "q_source"], as_index=False)
           .agg(bit_accuracy=("bit_accuracy", "mean"),
                w_star=("w_cos", "median"),
                R10_w0=("R10_w0", "mean"),
                gain=("gain", "mean"),
                gain_sd=("gain", "std"),
                runs=("gain", "count"))
           .sort_values(["predictor", "bit_accuracy"], ascending=[True, False]))
summary.style.hide(axis="index").format({
    "bit_accuracy": "{:.4f}", "R10_w0": "{:.4f}",
    "gain": "{:+.4f}", "gain_sd": "{:.4f}"})


In [ ]:
head_rows = summary[summary.corruption >= 0]
anchor = summary[summary.corruption < 0]
STYLE = {"v_ref": ("s--", "tab:orange"), "cpas": ("o-", "tab:blue")}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.6))
for src, (fmt, colour) in STYLE.items():
    h = head_rows[head_rows.q_source == src]
    a = anchor[anchor.q_source == src]
    ax1.plot(h.bit_accuracy, h.w_star, fmt, color=colour, label=f"q = {src}")
    ax2.plot(h.bit_accuracy, h.gain, fmt, color=colour, label=f"q = {src}")
    # The anchor is a different model, not a corrupted head: marked apart so it
    # is never read as a sample of the synthetic curve.
    ax1.plot(a.bit_accuracy, a.w_star, "*", ms=14, color=colour)
    ax2.plot(a.bit_accuracy, a.gain, "*", ms=14, color=colour)

ax1.set_yscale("log")
ax1.set_xlabel("per-bit accuracy of the predictor")
ax1.set_ylabel("selected $w$ (median over seeds)")
ax1.set_title("The cosine weight is a fallback", fontsize=10)
ax2.axhline(0, lw=0.8, color="0.6")
ax2.set_xlabel("per-bit accuracy of the predictor")
ax2.set_ylabel("val R@10 gain over $w = 0$")
ax2.set_title("What the fallback is worth", fontsize=10)
for ax in (ax1, ax2):
    ax.invert_xaxis(); ax.legend(fontsize=8)
fig.text(0.5, -0.06, "★ = linear probe, a genuinely weaker predictor "
         "rather than a corrupted one", ha="center", fontsize=8, color="0.35")
fig.tight_layout(); plt.show()


Both predictions hold. The selected $w$ grows monotonically as the code
degrades — from 1 at the delivered predictor to 300 at 0.70 bit accuracy — and
it does so consistently across every one of the 9 runs per level (3 corruption
draws × 3 combiner seeds).

**The two query vectors are not interchangeable.** The gain over $w = 0$ is
$2$–$6\times$ larger for the composed query at every level, and the gap widens
monotonically:

| per-bit accuracy | R@10 at $w=0$ | $w^\star$ | gain, $q = v_{ref}$ | gain, $q = $ CPAS-MLP |
|---|---|---|---|---|
| 0.917 (delivered) | 0.543 | 1 | +0.004 | **+0.005** ± 0.001 |
| 0.895 | 0.437 | 10 | +0.013 | **+0.031** ± 0.018 |
| 0.874 | 0.383 | 10 | +0.007 | **+0.053** ± 0.006 |
| 0.830 | 0.255 | 100 | +0.030 | **+0.120** ± 0.009 |
| 0.787 | 0.182 | 100 | +0.042 | **+0.184** ± 0.017 |
| 0.700 | 0.089 | 300 | +0.086 | **+0.256** ± 0.013 |
| *linear probe, 0.908* | *0.459* | *1* | *−0.009* | *+0.005* |

At 0.70 bit accuracy attribute matching has collapsed to R@10 0.089; the fusion
module restores it to 0.345, a **3.9× recovery**. The standard deviations are an
order of magnitude below the effects, so the ordering is not a seed artefact.

**What $w$ actually is.** That plateau is not a coincidence: CPAS-MLP scores
0.361 on its own in pure cosine space (§7.1). As $w$ grows the first term stops
mattering and the score *degenerates into cosine retrieval with the composed
query*. So $w$ is not a weight among weights — it is **the dial between two
retrieval regimes**, attribute matching at one end and embedding retrieval at
the other, and the fusion module is the floor the system lands on when the
attribute code stops being informative. The sweep picks the mixture.

This is why the delivered $w = 1$ should be read as **compliance-preserving
rather than performance-carrying**: at 0.917 bit accuracy the mixture is almost
entirely attribute matching, and Φ rides along at no measurable cost.

> **The synthetic curve overstates the case, and the anchor says so.** The
> linear probe (★) is a *genuinely* weaker predictor at 0.9084 bit accuracy,
> and it gains only **+0.005** from the composed query — the same as the clean
> head, and six times less than synthetic corruption of comparable severity
> (+0.031 at 0.8954). Uniform corruption produces errors *independent of image
> content*, so the embedding is genuinely complementary information; the probe's
> errors are **structured**, concentrated on the attributes CLIP itself
> represents poorly, so the embedding is correlated with the error and cannot
> correct it. What this experiment measures is therefore robustness **to noise**,
> not robustness **to a weaker model**. We report it as such.


## 9. Error decomposition and the ceiling

The score consumes two predicted objects — the reference's code and the
database codes. Substituting true labels on each side separately attributes the
remaining gap:

In [ ]:
label_probs = labels.float()
oracle = {"predicted probs": (head_probs, head_probs > head_thresholds),
          "true labels": (label_probs, labels)}

def decomposition_fn(db_side, ref_side):
    db_probs, db_code = oracle[db_side]
    ref_probs, ref_code_m = oracle[ref_side]
    def fn(entry, refs, pr, nr):
        return attribute_scores(db_probs, db_code,
                                target_code(ref_probs[refs], pr, nr), pr, nr,
                                lam_constraint=CFG["lam_constraint"])
    return fn

rows = []
for name, db_side, ref_side in [
        ("both predicted", "predicted probs", "predicted probs"),
        ("perfect reference code", "predicted probs", "true labels"),
        ("perfect database codes", "true labels", "predicted probs"),
        ("both perfect (oracle)", "true labels", "true labels")]:
    df = benchmark(decomposition_fn(db_side, ref_side), name)
    rows.append({"configuration": name,
                 "R@10": float(df[df["query"] == "MEAN"]["R@10"].iloc[0])})
pd.DataFrame(rows).style.format({"R@10": "{:.4f}"})

Three statements this table grounds:

- **Nothing is lost to composition or ranking.** With true codes on both sides
  the benchmark is solved exactly — the score is a faithful implementation of
  the criterion, and the entire remaining gap is attribute-prediction error.
- **Database-side error costs more than reference-side**: one wrong reference
  bit shifts a single target, while database errors corrupt every comparison.
- **The ceiling for this encoder is in sight.** §4.3 measured that 5.4× more
  training data buys +0.003 bit accuracy: frozen ViT-B/32 features are
  saturated. The single remaining lever is the encoder itself — §3.3 permits
  evaluating other models alongside the required one, and the decomposition
  predicts exactly what a stronger encoder would buy: everything between the
  first row and 1.0 that better codes can reach.

## 10. Negative results and limitations

Reported because they are measured, and because two of them contradict earlier
project documentation.

**Reliability-weighted Hamming does not pay.** Weighting each non-queried
attribute by the predictor's held-out Youden's J (informedness — accuracy is
the wrong weight on CelebA's imbalanced attributes, where a majority-class
predictor scores 88% on `Wearing_Necklace` while detecting nothing) gives
+0.014 R@10 alone — under the trained-comparison resolution — and *hurts* when
combined with the soft reference (MLP head: 0.567 → 0.554, measured with the
released runner via `--reliability-weights`). A mispredicted attribute
evidently still carries signal along with its noise; zeroing it discards both.
The knob remains in the codebase, default-off.

**λ is saturated.** $\lambda \in \{4, 16, 100\}$ produce identical validation
R@10 in every sweep configuration; only $\lambda = 1$ is worse. Earlier
documentation claimed $\lambda = 4$ *beat* $\lambda = 100$; with the current
predictors the constraint penalty acts as a threshold, not a tuned trade-off.
We keep $\lambda = 4$ as the smallest value on the plateau.

**Hyperparameter selection at the sweep's resolution.** The validation
benchmark evaluates ≤ 2,800 binary trials per configuration, i.e. a standard
error of ~0.009 on R@10 — margins below that (and several sweep margins are)
select noise. Where selection was unresolved we preferred the simpler setting;
the delivered $w = 1$ is justified in §8 on compliance grounds, not
performance.

**Benchmark structure.** `-Young` appears twice among the 14 mandatory queries
(the MEAN weights it double); the validation benchmark reuses the 14 query
*shapes*, so tuning is held out in images but not in query distribution; and
single-seed trained results (the combiner) carry the ±0.019 seed spread
measured on this architecture.

## 11. Conclusions

**Method.** Move the conditioning process into a learned attribute space:
predict all 40 attribute probabilities from frozen CLIP features, form the
composite query as the reference's predicted code with the queried bits forced
— the provably optimal target under the assignment's own ground-truth rule —
and rank by a dynamic similarity metric whose terms implement that rule's two
conditions, plus a cosine term carrying the trained fusion module's composite
embedding.

**Results.** 0.106 → 0.564 R@10 (5.3×) over the required zero-shot baseline,
with every intermediate step measured: in-space directions (+0.10), trained
fusion in cosine space (+0.14), criterion-space ranking (+0.20), soft reference
(+0.04). The error decomposition attributes the entire remaining gap to
attribute prediction, whose accuracy is saturated for this encoder — a clean
ceiling statement rather than an open tuning problem.

**Assignment compliance, explicitly.** The fusion module Φ (§1 of the
assignment) is CPAS-MLP: it natively accepts the visual reference and multiple
signed textual conditions, produces a composite query embedding, and is present
in the delivered ranking through the cosine term. The *dynamic similarity
metric* is the §3 score: positive and negative constraints enter through
different terms with different signs, the identity requirement enters as the
distance the assignment itself defines, and §8 measures — rather than asserts —
the regime in which each component carries the ranking. Every result is
reported with and without each component.

### References

- Berasi, Farina, Mancini, Ricci, Strisciuglio. *Not Only Text: Exploring
  Compositionality of Visual Representations in VLMs.* CVPR 2025.
- Lim, Hyoseok, Park, Oh. *CLAY: Conditional Visual Similarity Modulation in
  Vision-Language Embedding Space.* CVPR 2026.
- Liu, Luo, Wang, Tang. *Deep Learning Face Attributes in the Wild.* ICCV 2015.
- Radford et al. *Learning Transferable Visual Models From Natural Language
  Supervision.* ICML 2021.
- van den Oord, Li, Vinyals. *Representation Learning with Contrastive
  Predictive Coding.* arXiv 2018.
- Youden. *Index for rating diagnostic tests.* Cancer 1950.